# Visual Asset Auditing System — Production Integration Notebook

This notebook is a readable, backend-handoff implementation of the Visual Asset Auditing System. It uses named functions rather than cell numbers, keeps AlloyDB retrieval read-only, supports text/reference-image audits, and applies a 3 GB App Engine memory profile.

Run the notebook from top to bottom for initial validation. Backend code should call `run_visual_audit(...)` directly.


## Package installation

The installation cell pins compatible major versions so a future SDK release cannot silently break the backend. Restart the Colab runtime only if pip reports that an already-imported package was replaced.

In [ ]:
# Install the production dependency set used by this notebook.
!pip install -q "google-genai>=1.52,<3.0" "google-cloud-alloydb-connector[asyncpg]>=1.10,<2" "google-cloud-storage>=2,<4" "sqlalchemy[asyncio]>=2,<3" "pgvector>=0.3,<1" "kneed>=0.8,<1" "pandas>=2,<4" "numpy>=1.26,<3" "pillow>=10,<13" "opencv-python-headless>=4.10,<5" "pydantic>=2,<3"

## Google Cloud Authentication

Colab uses interactive Google Cloud authentication. App Engine uses Application Default Credentials from its service account, so the backend does not run a browser sign-in flow.


In [ ]:
# Authenticate with Google Cloud
# Colab uses its interactive sign-in. App Engine uses Application Default Credentials.
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated with Google Cloud from Colab.")
except ImportError:
    print("Using Application Default Credentials supplied by the deployment runtime.")

import google.genai as genai
from google.genai import types

print("Google GenAI SDK imported successfully.")


## Runtime configuration

Set these values in Colab Secrets or the App Engine environment: `PROJECT_ID`, `REGION`, `ALLOYDB_CLUSTER`, `ALLOYDB_INSTANCE`, `DB_USER`, `DB_NAME`, and `DB_SCHEMA`. Set `DB_PASSWORD` unless IAM database authentication is enabled.

Optional production settings include `VERTEX_LOCATION`, `DB_IP_TYPE` (`PUBLIC`, `PRIVATE`, or `PSC`), model names, `VERIFY_DATABASE_ON_STARTUP`, `LOAD_LIVE_TAG_VOCABULARY`, `ALLOW_DATABASE_WRITES`, and the timeout/size limits below. No secret is embedded in this notebook.

In [ ]:
# Production runtime configuration — Colab Secrets or environment variables only
import os
import re


def _read_runtime_setting(name: str, default=None, required: bool = False):
    value = os.getenv(name)
    if value in (None, ""):
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    if value in (None, ""):
        value = default
    if required and value in (None, ""):
        raise RuntimeError(
            f"Missing required runtime setting: {name}. "
            "Add it to Colab Secrets or the App Engine environment before continuing."
        )
    return value


def _read_bool_setting(name: str, default: bool = False) -> bool:
    value = _read_runtime_setting(name, str(default))
    return str(value).strip().lower() in {"1", "true", "yes", "on"}


def _read_int_setting(name: str, default: int, minimum: int, maximum: int) -> int:
    try:
        value = int(_read_runtime_setting(name, default))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} must be an integer") from exc
    if not minimum <= value <= maximum:
        raise ValueError(f"{name} must be between {minimum} and {maximum}")
    return value


PROJECT_ID = _read_runtime_setting("PROJECT_ID", required=True)
REGION = _read_runtime_setting("REGION", required=True)
VERTEX_LOCATION = _read_runtime_setting("VERTEX_LOCATION", REGION)
ALLOYDB_CLUSTER = _read_runtime_setting("ALLOYDB_CLUSTER", required=True)
ALLOYDB_INSTANCE = _read_runtime_setting("ALLOYDB_INSTANCE", required=True)
DB_USER = _read_runtime_setting("DB_USER", required=True)
DB_ENABLE_IAM_AUTH = _read_bool_setting("DB_ENABLE_IAM_AUTH", False)
DB_PASSWORD = _read_runtime_setting("DB_PASSWORD", required=not DB_ENABLE_IAM_AUTH)
DB_NAME = _read_runtime_setting("DB_NAME", required=True)
DB_SCHEMA = str(_read_runtime_setting("DB_SCHEMA", required=True))
DB_IP_TYPE = str(_read_runtime_setting("DB_IP_TYPE", "PUBLIC")).upper()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", DB_SCHEMA):
    raise ValueError("DB_SCHEMA must be a simple PostgreSQL identifier")

# Model names remain configurable so the backend can promote models without code edits.
GEMINI_ORCHESTRATOR_MODEL = _read_runtime_setting("GEMINI_ORCHESTRATOR_MODEL", "gemini-2.5-flash")
GEMINI_INFERENCE_MODEL = _read_runtime_setting("GEMINI_INFERENCE_MODEL", "gemini-2.5-flash")
EMBEDDING_MODEL = _read_runtime_setting("EMBEDDING_MODEL", "gemini-embedding-2-preview")

GENAI_TIMEOUT_MS = _read_int_setting("GENAI_TIMEOUT_MS", 90_000, 10_000, 300_000)
GCS_DOWNLOAD_TIMEOUT_SECONDS = _read_int_setting("GCS_DOWNLOAD_TIMEOUT_SECONDS", 30, 5, 120)
DB_QUERY_TIMEOUT_SECONDS = _read_int_setting("DB_QUERY_TIMEOUT_SECONDS", 90, 10, 300)
MAX_SOURCE_BYTES = _read_int_setting("MAX_SOURCE_BYTES", 48 * 1024 * 1024, 1 * 1024 * 1024, 128 * 1024 * 1024)
VERIFY_DATABASE_ON_STARTUP = _read_bool_setting("VERIFY_DATABASE_ON_STARTUP", False)
LOAD_LIVE_TAG_VOCABULARY = _read_bool_setting("LOAD_LIVE_TAG_VOCABULARY", False)
ALLOW_DATABASE_WRITES = _read_bool_setting("ALLOW_DATABASE_WRITES", False)

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=VERTEX_LOCATION,
    http_options=types.HttpOptions(timeout=GENAI_TIMEOUT_MS),
)

# Used by default instead of scanning millions of rows for a vocabulary on each request.
WEB_AUDIT_VISION_TAGS = [
    "Logo", "Brand", "Icon", "Symbol", "Font", "Text", "Product", "Screenshot",
    "Web page", "Graphic design", "Illustration", "Portrait", "Person", "Face",
    "Technology", "Mobile phone", "Computer", "Button", "Banner", "Advertising",
]

print(
    "Runtime configured: "
    f"project={PROJECT_ID}, vertex_location={VERTEX_LOCATION}, schema={DB_SCHEMA}, "
    f"inference_model={GEMINI_INFERENCE_MODEL}, writes_enabled={ALLOW_DATABASE_WRITES}"
)

In [ ]:
# AlloyDB connection setup — current connector API and a 3 GB App Engine-safe pool
import asyncio
import asyncpg
from typing import Tuple
from sqlalchemy.ext.asyncio import create_async_engine, AsyncEngine
from google.cloud.alloydbconnector import IPTypes, AsyncConnector

_engine_cache = {}
_connector_cache = {}


def _configured_ip_type():
    supported = {
        "PUBLIC": IPTypes.PUBLIC,
        "PRIVATE": IPTypes.PRIVATE,
        "PSC": IPTypes.PSC,
    }
    if DB_IP_TYPE not in supported:
        raise ValueError(f"Unsupported DB_IP_TYPE={DB_IP_TYPE!r}; use PUBLIC, PRIVATE, or PSC.")
    return supported[DB_IP_TYPE]


async def get_alloydb_connection(reuse: bool = True) -> Tuple[AsyncEngine, AsyncConnector]:
    """Return a small pooled AlloyDB engine suitable for a 3 GB App Engine instance."""
    if reuse and "default" in _engine_cache:
        return _engine_cache["default"], _connector_cache["default"]

    connector = AsyncConnector(refresh_strategy="lazy")

    async def getconn():
        instance_uri = ALLOYDB_INSTANCE
        if not instance_uri.startswith("projects/"):
            instance_uri = (
                f"projects/{PROJECT_ID}/locations/{REGION}/clusters/"
                f"{ALLOYDB_CLUSTER}/instances/{ALLOYDB_INSTANCE}"
            )
        try:
            return await asyncio.wait_for(
                connector.connect(
                    instance_uri,
                    "asyncpg",
                    user=DB_USER,
                    password=DB_PASSWORD,
                    db=DB_NAME,
                    enable_iam_auth=DB_ENABLE_IAM_AUTH,
                    ip_type=_configured_ip_type(),
                ),
                timeout=10.0,
            )
        except asyncio.TimeoutError as exc:
            raise ConnectionError(
                f"AlloyDB connection timed out after 10 seconds: {instance_uri}. "
                "Check App Engine VPC/public connectivity and authorized networking."
            ) from exc
        except Exception as exc:
            raise ConnectionError(f"Failed to connect to AlloyDB: {exc}") from exc

    engine = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        echo=False,
        pool_size=3,
        max_overflow=2,
        pool_timeout=10,
        pool_recycle=1800,
        pool_pre_ping=True,
    )
    if reuse:
        _engine_cache["default"] = engine
        _connector_cache["default"] = connector
    return engine, connector


async def close_alloydb_connection() -> None:
    """Release pooled connections and the connector during backend shutdown."""
    engines = list(_engine_cache.values())
    connectors = list(_connector_cache.values())
    _engine_cache.clear()
    _connector_cache.clear()
    for engine in engines:
        await engine.dispose()
    for connector in connectors:
        await connector.close()

In [ ]:
# Database connection verification — opt-in, read-only, and constant-time
async def verify_database_connection() -> bool:
    """Confirm that AlloyDB and visual_assets are reachable without scanning the table."""
    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        await db.fetchval(f"SELECT 1 FROM {DB_SCHEMA}.visual_assets LIMIT 1")
    print(f"Connected to AlloyDB. Read-only access to {DB_SCHEMA}.visual_assets is ready.")
    return True


# Importing the notebook as backend code must never open a network connection.
DATABASE_CONNECTION_READY = False
if VERIFY_DATABASE_ON_STARTUP:
    DATABASE_CONNECTION_READY = await verify_database_connection()
else:
    print("Database verification deferred. Run `await verify_database_connection()` when desired.")

In [ ]:
# 1. Audit configuration and multimodal query embedding
import asyncio
import json
import mimetypes
import re
import time
from typing import Any, Dict, List, Optional, Tuple
from pydantic import BaseModel, ConfigDict, Field


class AuditContextModel(BaseModel):
    """Strict, versioned contract between the configuration model and retrieval."""
    model_config = ConfigDict(extra="forbid")

    audit_goal: str = Field(min_length=3, description="Refined, precise version of the user's goal.")
    image_description: Optional[str] = Field(None, description="Visible reference-image evidence, or null.")
    reference_is_composite_canvas: bool = Field(description="Whether the reference is a larger scene containing the target.")
    inclusion_criteria: List[str] = Field(min_length=1, max_length=8, description="Specific, testable match requirements.")
    exclusion_criteria: List[str] = Field(min_length=1, max_length=8, description="Specific rejection requirements.")
    adjudication_logic: str = Field(min_length=10, description="A clear IF-THEN-ELSE pass/fail rule.")
    search_keywords: List[str] = Field(min_length=1, max_length=12, description="Broad single-token retrieval terms.")
    vision_tag_filter: List[str] = Field(default_factory=list, max_length=8, description="Terms drawn only from the supplied Vision tag vocabulary.")
    audit_instructions: str = Field(min_length=10, description="Grounded instructions for the final visual judge.")
    extraction_schema: Dict[str, Any] = Field(default_factory=dict, description="Additional typed properties to extract; never verdict/confidence/rationale fields.")


def get_image_mime_type(path: str) -> str:
    mime, _ = mimetypes.guess_type(str(path).split("?", 1)[0])
    return mime if mime and mime.startswith("image/") else "image/jpeg"


def _call_with_retry(callable_, attempts: int = 3, base_delay: float = 0.8):
    """Retry transient SDK/network failures without multiplying active workers."""
    last_error = None
    for attempt in range(attempts):
        try:
            return callable_()
        except (ValueError, TypeError):
            raise
        except Exception as exc:
            last_error = exc
            if attempt + 1 < attempts:
                time.sleep(base_delay * (2 ** attempt))
    raise RuntimeError(f"Remote model call failed after {attempts} attempts: {last_error}") from last_error


def _normalize_audit_config(config: Dict[str, Any], user_goal: str, active_tags: List[str]) -> Dict[str, Any]:
    """Deterministically enforce retrieval-safe values instead of trusting prompt compliance."""
    token_sources = list(config.get("search_keywords", []) or []) + [user_goal]
    keywords: List[str] = []
    seen = set()
    for source in token_sources:
        for token in re.findall(r"[A-Za-z0-9][A-Za-z0-9_-]*", str(source)):
            key = token.casefold()
            if len(token) >= 2 and key not in seen:
                seen.add(key)
                keywords.append(token)
            if len(keywords) == 12:
                break
        if len(keywords) == 12:
            break
    config["search_keywords"] = keywords or ["image"]

    allowed = {str(tag).casefold(): str(tag) for tag in active_tags}
    normalized_tags = []
    for tag in config.get("vision_tag_filter", []) or []:
        canonical = allowed.get(str(tag).casefold())
        if canonical and canonical not in normalized_tags:
            normalized_tags.append(canonical)
    config["vision_tag_filter"] = normalized_tags[:8]

    forbidden = {"matches_criteria", "match_confidence", "match_rationale", "visual_analysis_step_by_step"}
    config["extraction_schema"] = {
        str(key): value for key, value in (config.get("extraction_schema") or {}).items()
        if str(key) not in forbidden
    }
    return config


def generate_audit_config(
    user_goal: str,
    reference_image_description: Optional[str] = None,
    available_tags: Optional[List[str]] = None,
) -> Dict[str, Any]:
    """Translate a user goal into a validated, visually grounded audit contract."""
    active_tags = list(available_tags or WEB_AUDIT_VISION_TAGS)
    prompt = f"""
You are configuring an enterprise visual-asset audit. Return only data matching the supplied JSON schema.

USER GOAL:
{user_goal}

VISIBLE REFERENCE-IMAGE ANALYSIS (may be absent):
{reference_image_description or 'No reference image was supplied.'}

Rules:
1. Refine the goal without changing its meaning. Never invent a brand, text, shape, version, or compliance fact that is not in the goal or visible-reference analysis.
2. Decide whether the reference is a standalone target or a composite canvas containing the target.
3. Write testable inclusion and exclusion criteria. Preserve visible solid-versus-gradient, color, spelling, typography, shape, crop, blur, and layout evidence whenever it distinguishes versions or identities.
4. A cropped, blurred, resized, partially occluded, or small embedded target remains eligible when its identity-defining evidence is still visible. Do not demand the reference background or surrounding scene.
5. For exact/same-image goals, require identity with the supplied image while permitting compression, resizing, crop, blur, and embedding inside a larger canvas. For similar-image goals, describe the allowed variation.
6. Write explicit IF-THEN-ELSE adjudication logic. Exclude related products, look-alikes, misspellings, and visibly different versions when the goal is version-specific.
7. `search_keywords` must be broad single tokens. `vision_tag_filter` may contain only exact entries from this list: {active_tags}
8. `extraction_schema` may define useful boolean/integer/number/string evidence, but must not define matches_criteria, match_confidence, match_rationale, or visual_analysis_step_by_step.
"""

    response = _call_with_retry(lambda: client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_json_schema=AuditContextModel.model_json_schema(),
            temperature=0.0,
        ),
    ))
    parsed = AuditContextModel.model_validate_json(response.text).model_dump()
    return _normalize_audit_config(parsed, user_goal, active_tags)


def build_fused_query_text(context_dict: Dict[str, Any], is_negative: bool = False) -> str:
    if is_negative:
        return "Exclude images that: " + "; ".join(context_dict.get("exclusion_criteria", []) or [])
    parts = [
        f"Audit goal: {context_dict.get('audit_goal', '')}",
        "Required visual evidence: " + "; ".join(context_dict.get("inclusion_criteria", []) or []),
    ]
    if context_dict.get("image_description"):
        parts.append(f"Reference-image evidence: {context_dict['image_description']}")
    return " | ".join(parts)


async def embed_audit_context(context_dict: Dict[str, Any], reference_image_path: Optional[str] = None) -> Tuple[List[float], List[float]]:
    """Generate positive and negative 768-dimensional retrieval embeddings."""
    positive: List[Any] = []
    if reference_image_path:
        mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            positive.append(types.Part.from_uri(file_uri=reference_image_path, mime_type=mime))
        else:
            with open(reference_image_path, "rb") as source:
                payload = source.read(MAX_SOURCE_BYTES + 1)
            if len(payload) > MAX_SOURCE_BYTES:
                raise ValueError(f"Reference image exceeds MAX_SOURCE_BYTES={MAX_SOURCE_BYTES}")
            positive.append(types.Part.from_bytes(data=payload, mime_type=mime))
    positive.append(build_fused_query_text(context_dict)[:3_000])
    negative = [build_fused_query_text(context_dict, is_negative=True)[:1_500]]

    def embed(contents: List[Any]) -> List[float]:
        result = _call_with_retry(lambda: client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=contents,
            config=types.EmbedContentConfig(output_dimensionality=768),
        ))
        vector = list(result.embeddings[0].values)
        if len(vector) != 768:
            raise ValueError(f"Expected a 768-dimensional embedding, received {len(vector)}")
        return vector

    return tuple(await asyncio.gather(asyncio.to_thread(embed, positive), asyncio.to_thread(embed, negative)))

## Retrieval, Deterministic Reranking, and Recall Segmentation

The production runtime combines vector, full-text, tag, and exact-hash evidence with reciprocal-rank fusion. Reranking is deterministic and in-memory; it does **not** call an LLM text cross-encoder. Drop-off segmentation keeps a low-score reserve so crop, blur, and small-target evidence can be rescued before the final visual audit.


In [ ]:
# 2. Parallel Hybrid Search with RRF & Drop-Off Detection
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
from kneed import KneeLocator
from pgvector.asyncpg import register_vector
import asyncio
from concurrent.futures import ThreadPoolExecutor

async def run_semantic_reranking_and_filter(
    df_high: pd.DataFrame,
    df_edge: pd.DataFrame,
    audit_context: dict,
    max_workers: int = 2,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Retain deterministic retrieval ranks without an LLM text-reranking call.

    The production runtime later adds bounded local reference-image verification,
    but this base function is safe to call independently and performs no network I/O.
    """
    del audit_context, max_workers

    def _sorted(frame: pd.DataFrame) -> pd.DataFrame:
        if frame is None or frame.empty:
            return pd.DataFrame()
        sort_column = "relevance_score" if "relevance_score" in frame.columns else None
        if sort_column:
            return frame.sort_values(sort_column, ascending=False).reset_index(drop=True)
        return frame.reset_index(drop=True)

    print("Deterministic reranking: no LLM text cross-encoder call.")
    return _sorted(df_high), _sorted(df_edge), pd.DataFrame()


def compute_weighted_rrf_rerank(candidates: list, audit_context: dict) -> list:
    """Zero-latency in-memory multi-factor reranker. Executes in local CPU RAM (< 0.5ms) without external API overhead."""
    tag_filter = [t.lower().strip() for t in audit_context.get("vision_tag_filter", []) if t]
    search_keywords = [k.lower().strip() for k in audit_context.get("search_keywords", []) if k]
    reranked = []

    for item in candidates:
        row = item["data"]
        base_score = item["score"]
        multiplier = 1.0

        # 1. Vision tag exact hit boost (2.0x per matching tag up to 8x)
        tags = [str(t).lower() for t in (row.get("vision_tags") or [])]
        tag_hits = sum(1 for t in tags if any(ft in t or t in ft for ft in tag_filter))
        if tag_hits > 0:
            multiplier *= (2.0 ** min(tag_hits, 3))

        # 2. Keyword exact hit in filename or description boost (1.5x per matching keyword up to 2.25x)
        desc = str(row.get("gemini_description") or "").lower()
        fname = str(row.get("asset_filename") or "").lower()
        kw_hits = sum(1 for kw in search_keywords if kw in desc or kw in fname)
        if kw_hits > 0:
            multiplier *= min(1.5 ** kw_hits, 2.25)

        reranked.append({
            "data": row,
            "score": base_score * multiplier,
            "base_rrf_score": base_score,
            "rerank_multiplier": multiplier,
            "vector_distance": item.get("vector_distance", 1.0)
        })

    reranked.sort(key=lambda x: x["score"], reverse=True)
    return reranked

async def run_hybrid_search(scope_config: dict, audit_context: dict, reference_image_path: Optional[str] = None, limit: int = 10000) -> List[dict]:
    """Performs parallel 3-Arm Vector + Keyword + Tag Boosting search with RRF fusion, local reranking, and deduplication (No silent cliff)."""
    pos_vec, neg_vec = await embed_audit_context(audit_context, reference_image_path)

    engine, _ = await get_alloydb_connection()

    search_keywords = audit_context.get("search_keywords", [])
    keyword_query_str = " OR ".join(search_keywords) if search_keywords else ""

    tag_filter = audit_context.get("vision_tag_filter", [])
    tag_filter = tag_filter if tag_filter else []

    # Run the 3 database query arms concurrently using pooled SQLAlchemy connections.
    async def run_vector():
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            await register_vector(db)
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                       (embedding <=> $1::vector) as vector_distance
                FROM {DB_SCHEMA}.visual_assets
                ORDER BY $1::vector <=> embedding - (0.3 * (embedding <=> $2::vector)) ASC
                LIMIT $3
                """,
                pos_vec, neg_vec, limit
            )
            return [dict(r) for r in rows]

    async def run_fts():
        if not keyword_query_str:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                FROM {DB_SCHEMA}.visual_assets
                WHERE to_tsvector('english', gemini_description) @@ websearch_to_tsquery('english', $1)
                LIMIT $2
                """,
                keyword_query_str, limit
            )
            return [dict(r) for r in rows]

    async def run_tags():
        if not tag_filter:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                FROM {DB_SCHEMA}.visual_assets
                WHERE EXISTS (
                    SELECT 1 FROM unnest(vision_tags) tag
                    WHERE EXISTS (
                        SELECT 1 FROM unnest($1::text[]) filter_tag
                        WHERE tag ILIKE '%' || filter_tag || '%'
                    )
                )
                LIMIT $2
                """,
                tag_filter, limit
            )
            return [dict(r) for r in rows]

    vector_task = run_vector()
    fts_task = run_fts()
    tag_task = run_tags()

    vector_results, fts_results, tag_results = await asyncio.gather(vector_task, fts_task, tag_task)

    promoted_ids = set()
    for r in fts_results[:100]:
        promoted_ids.add(str(r["asset_id"]))
    for r in tag_results[:100]:
        promoted_ids.add(str(r["asset_id"]))

    # 3-Arm RRF Fusion (k=60)
    k = 60
    results_map = {}

    # Pre-populate with vector distances
    vector_distance_map = {str(r["asset_id"]): r["vector_distance"] for r in vector_results}

    def upsert_ranks(results_list, weight=1.0):
        for rank, row in enumerate(results_list):
            img_id = str(row["asset_id"])
            if img_id not in results_map:
                results_map[img_id] = {"data": row, "score": 0.0, "vector_distance": vector_distance_map.get(img_id, 1.0)}
            results_map[img_id]["score"] += weight / (k + rank + 1)

    upsert_ranks(vector_results, weight=0.60)
    if fts_results:
        upsert_ranks(fts_results, weight=0.25)
    if tag_results:
        upsert_ranks(tag_results, weight=0.15)

    fused = list(results_map.values())

    # Zero-Latency In-Memory Reranking (Provides a smooth score curve for Kneedle)
    reranked_fused = compute_weighted_rrf_rerank(fused, audit_context)

    # Deduplication
    seen_identifiers = set()
    deduplicated = []
    for item in reranked_fused:
        row = item["data"]
        img_id = str(row["asset_id"])
        img_identifier = row.get("content_hash") or row.get("gcs_raw_path")
        if img_identifier not in seen_identifiers:
            seen_identifiers.add(img_identifier)
            is_promoted = img_id in promoted_ids
            deduplicated.append({
                **row,
                "relevance_score": item["score"],
                "base_rrf_score": item.get("base_rrf_score", 0),
                "rerank_multiplier": item.get("rerank_multiplier", 1),
                "promoted_by_keyword_or_tag": is_promoted,
                "vector_distance": item.get("vector_distance", 1.0)
            })

    return deduplicated[:limit]

def detect_dropoff_flawless(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Apply curvature plus rolling volatility to label High, Borderline, and Low bands."""
    if sensitivity <= 0:
        raise ValueError("sensitivity must be greater than zero")
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_sorted = df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)
    y = df_sorted["relevance_score"].values
    x = np.arange(len(y))

    y_min, y_max = y.min(), y.max()
    if y_max == y_min:
        return df_sorted.iloc[:int(len(y)*0.3)], df_sorted.iloc[int(len(y)*0.3):int(len(y)*0.6)], df_sorted.iloc[int(len(y)*0.6):]

    y_norm = (y - y_min) / (y_max - y_min + 1e-9)
    x_norm = x / (len(x) - 1)

    coords = np.column_stack((x_norm, y_norm))
    line_start, line_end = coords[0], coords[-1]
    line_vec = line_end - line_start
    line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
    vec_from_start = coords - line_start
    scalar_proj = np.dot(vec_from_start, line_vec_norm)
    proj_on_line = line_start + np.outer(scalar_proj, line_vec_norm)
    dist_to_line = np.sqrt(np.sum((coords - proj_on_line)**2, axis=1))

    idx1 = np.argmax(dist_to_line)

    window = max(3, int(len(y) * 0.05))
    rolling_std = pd.Series(y_norm).rolling(window=window, center=True).std().fillna(0).values
    noise_threshold = np.mean(rolling_std) * (0.6 / sensitivity)

    idx2 = len(y) - 1
    for i in range(idx1 + 2, len(rolling_std)):
        if rolling_std[i] < noise_threshold:
            idx2 = i
            break

    min_borderline_width = max(10, int((len(y) - idx1) * 0.25))
    if (idx2 - idx1) < min_borderline_width:
        idx2 = min(len(y) - 1, idx1 + min_borderline_width)

    idx1 = max(10, idx1)

    high_df = df_sorted.iloc[:idx1 + 1].copy()
    edge_df = df_sorted.iloc[idx1 + 1: idx2 + 1].copy()
    low_df = df_sorted.iloc[idx2 + 1:].copy()

    # Visual Vector Safeguard: Check if any candidate has extremely high similarity (distance < 0.28)
    # even if it is currently classified in Low_df (or has been discarded).
    # Force rescue these to protect visual recall.
    if "vector_distance" in low_df.columns:
        rescued_vec = low_df[low_df["vector_distance"] < 0.28].copy()
        if not rescued_vec.empty:
            edge_df = pd.concat([edge_df, rescued_vec], ignore_index=True)
            low_df = low_df[low_df["vector_distance"] >= 0.28].copy()
            print(f"🛡️ Vector Safeguard triggered in Kneedle: Force-rescued {len(rescued_vec)} candidate(s) from Low to Borderline based on high visual similarity.")

    if "promoted_by_keyword_or_tag" in low_df.columns:
        rescued = low_df[low_df["promoted_by_keyword_or_tag"] == True].copy()
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued], ignore_index=True)
            low_df = low_df[low_df["promoted_by_keyword_or_tag"] != True].copy()
            print(f"🛡️ Safeguard triggered: Promoted {len(rescued)} composite/diluted candidates from Low to Borderline tier based on exact keyword/tag match.")

    return high_df, edge_df, low_df

In [ ]:
# 3. Hydration and bounded parallel visual audit inference
import asyncio
import io
import json
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from google.cloud import storage
from google.genai import types
from PIL import Image, ImageOps
from pydantic import ConfigDict, Field, create_model


def process_transparency(image_bytes: bytes, default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
    """Normalize a bounded image to RGB PNG and make transparent marks visible."""
    with Image.open(io.BytesIO(image_bytes)) as source:
        if source.width * source.height > 64_000_000:
            raise ValueError("Image exceeds the 64 MP processing safety limit")
        source = ImageOps.exif_transpose(source)
        try:
            source.draft("RGB", (1600, 1600))
        except Exception:
            pass
        source.thumbnail((1600, 1600), getattr(Image, "Resampling", Image).LANCZOS)
        if source.mode in ("RGBA", "LA") or (source.mode == "P" and "transparency" in source.info):
            rgba = source.convert("RGBA")
            background = Image.new("RGBA", rgba.size, default_bg + (255,))
            normalized = Image.alpha_composite(background, rgba).convert("RGB")
        else:
            normalized = source.convert("RGB")
        output = io.BytesIO()
        normalized.save(output, format="PNG", optimize=True)
        return output.getvalue()


def enforce_calibrated_precision_rules(df: pd.DataFrame, minimum_match_confidence: int = 75) -> pd.DataFrame:
    """Apply a conservative output threshold; this is a policy, not a zero-error claim."""
    if df.empty:
        return df

    def guardrail(row: pd.Series) -> pd.Series:
        confidence = int(row.get("match_confidence", 0) or 0)
        if bool(row.get("matches_criteria", False)) and confidence < minimum_match_confidence:
            row["matches_criteria"] = False
            row["match_rationale"] = (
                f"[CALIBRATED DEMOTION: confidence {confidence}% < {minimum_match_confidence}%] "
                + str(row.get("match_rationale", ""))
            )
        return row

    return df.apply(guardrail, axis=1)


# Backward-compatible name used by the production runtime cell.
enforce_zero_false_positives_rules = enforce_calibrated_precision_rules
_audit_storage_client = None


def _download_bounded(path: str) -> bytes:
    global _audit_storage_client
    if path.startswith("gs://"):
        bucket_name, blob_name = path[5:].split("/", 1)
        if _audit_storage_client is None:
            _audit_storage_client = storage.Client(project=PROJECT_ID)
        payload = _audit_storage_client.bucket(bucket_name).blob(blob_name).download_as_bytes(
            start=0,
            end=MAX_SOURCE_BYTES,
            timeout=GCS_DOWNLOAD_TIMEOUT_SECONDS,
        )
    else:
        with open(path, "rb") as source:
            payload = source.read(MAX_SOURCE_BYTES + 1)
    if len(payload) > MAX_SOURCE_BYTES:
        raise ValueError(f"Image exceeds MAX_SOURCE_BYTES={MAX_SOURCE_BYTES}")
    return payload


async def run_llm_audit_single(
    asset_data: Dict[str, Any],
    audit_config: Dict[str, Any],
    _executor=None,
    reference_image_part: Optional[types.Part] = None,
) -> Dict[str, Any]:
    """Hydrate and visually adjudicate one candidate against a strict dynamic schema."""
    extraction_schema = audit_config.get("extraction_schema", {}) or {}
    inclusion = audit_config.get("inclusion_criteria", []) or []
    exclusion = audit_config.get("exclusion_criteria", []) or []
    is_composite = bool(audit_config.get("reference_is_composite_canvas", False))

    fields = {
        "visual_analysis_step_by_step": (
            str,
            Field(description="Concise observable visual evidence: target location, shapes, text, typography, color/style, crop/blur, and conflicts. Do not reveal hidden reasoning."),
        ),
        "matches_criteria": (bool, Field(description="True only when visible evidence satisfies the adjudication rule.")),
        "match_confidence": (int, Field(ge=0, le=100, description="Calibrated visual-match confidence from 0 to 100.")),
        "match_rationale": (str, Field(description="Concise evidence-based explanation of the verdict.")),
    }
    reserved = set(fields)
    for field_name, field_info in extraction_schema.items():
        if field_name in reserved or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", str(field_name)):
            continue
        description = f"Extracted value for {field_name}"
        declared = field_info
        if isinstance(field_info, dict):
            declared = field_info.get("field_type", "string")
            description = str(field_info.get("description", description))
        field_type = {"boolean": bool, "integer": int, "number": float}.get(str(declared).lower(), str)
        fields[str(field_name)] = (field_type, Field(description=description))

    DynamicAuditModel = create_model(
        "DynamicAuditModel",
        __config__=ConfigDict(extra="forbid"),
        **fields,
    )

    reference_instructions = ""
    if reference_image_part:
        if is_composite:
            reference_instructions = (
                "image_0 is a composite reference scene and image_1 is the candidate. "
                "Isolate only the target described in the audit configuration; do not require the surrounding reference scene."
            )
        else:
            reference_instructions = (
                "image_0 is the standalone reference and image_1 is the candidate. "
                "Compare their identity-defining visual evidence side by side."
            )

    color_rule = (
        "Color, solid-versus-gradient treatment, and rendition are discriminating evidence."
        if audit_config.get("_search_policy", {}).get("color_is_discriminating")
        else "Treat color as secondary unless an inclusion or exclusion criterion makes it discriminating."
    )
    prompt = f"""
Evaluate the candidate image against this visual-audit contract.
{reference_instructions}

Audit goal: {audit_config.get('audit_goal', '')}
Reference evidence: {audit_config.get('image_description') or 'None'}
Color policy: {color_rule}

Inclusion criteria:
{chr(10).join('- ' + str(item) for item in inclusion) or '- None'}

Exclusion criteria:
{chr(10).join('- ' + str(item) for item in exclusion) or '- None'}

Adjudication logic:
{audit_config.get('adjudication_logic', '')}

Additional instructions:
{audit_config.get('audit_instructions', '')}

Inspect the full candidate canvas, including corners and small embedded regions. Cropping, blur, compression, resizing, or partial occlusion do not by themselves cause failure when identity-defining evidence remains visible. Report observable evidence, extract the requested fields, then return the calibrated verdict.
"""

    loop = asyncio.get_running_loop()
    try:
        path = str(asset_data.get("gcs_raw_path") or "")
        if not path:
            raise ValueError("Candidate has no gcs_raw_path")
        payload = await loop.run_in_executor(_executor, _download_bounded, path)
        normalized = await loop.run_in_executor(_executor, process_transparency, payload)
        contents: List[Any] = []
        if reference_image_part:
            contents.append(reference_image_part)
        contents.extend([types.Part.from_bytes(data=normalized, mime_type="image/png"), prompt])

        def call_model():
            return _call_with_retry(lambda: client.models.generate_content(
                model=GEMINI_INFERENCE_MODEL,
                contents=contents,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_json_schema=DynamicAuditModel.model_json_schema(),
                    temperature=0.0,
                ),
            ))

        response = await loop.run_in_executor(_executor, call_model)
        extracted = DynamicAuditModel.model_validate_json(response.text).model_dump()
    except Exception as exc:
        extracted = {
            "matches_criteria": False,
            "match_confidence": 0,
            "match_rationale": f"Audit evaluation failed: {exc}",
            "visual_analysis_step_by_step": "Candidate could not be safely evaluated.",
            "error": str(exc),
        }
    return {**asset_data, **extracted}


async def run_llm_inference_on_dropoff_results(
    df_high: pd.DataFrame,
    df_edge: pd.DataFrame,
    audit_config: Dict[str, Any],
    max_workers: int = 2,
    reference_image_path: Optional[str] = None,
) -> pd.DataFrame:
    """Safe baseline runner; the production runtime cell adds batching and occurrence expansion."""
    candidates = pd.concat([df_high, df_edge], ignore_index=True).to_dict("records")
    if not candidates:
        return pd.DataFrame()
    workers = min(2, max(1, int(max_workers)))
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=workers) as executor:
        reference_part = None
        if reference_image_path:
            reference_payload = await loop.run_in_executor(executor, _download_bounded, reference_image_path)
            reference_png = await loop.run_in_executor(executor, process_transparency, reference_payload)
            reference_part = types.Part.from_bytes(data=reference_png, mime_type="image/png")
        results = []
        for start in range(0, len(candidates), 4):
            batch = candidates[start:start + 4]
            results.extend(await asyncio.gather(*[
                run_llm_audit_single(item, audit_config, _executor=executor, reference_image_part=reference_part)
                for item in batch
            ]))
    frame = enforce_calibrated_precision_rules(pd.DataFrame(results))
    return frame.sort_values("relevance_score", ascending=False, na_position="last").reset_index(drop=True)

## Calibration Summary and Optional Result Persistence

These functions generate the executive calibration summary and optionally persist final audit rows. Retrieval remains read-only against `visual_assets`; persistence is performed only when the backend explicitly calls the result-saving function.


In [ ]:
# 4. Optional persistence and bounded executive summary
async def save_audit_results_to_db(session_id: str, results_df: pd.DataFrame) -> None:
    """Persist results only when the deployment explicitly enables database writes."""
    if not ALLOW_DATABASE_WRITES:
        raise PermissionError("Database writes are disabled. Set ALLOW_DATABASE_WRITES=true to opt in.")
    if results_df is None or results_df.empty:
        return

    def json_default(value):
        if hasattr(value, "tolist"):
            return value.tolist()
        if pd.isna(value):
            return None
        return str(value)

    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw = await conn.get_raw_connection()
        db = raw.driver_connection
        async with db.transaction():
            for row in results_df.to_dict("records"):
                verdict = "PASS" if row.get("matches_criteria") is True else "FAIL"
                checks = {
                    key: value for key, value in row.items()
                    if key not in {"asset_id", "gcs_raw_path", "matches_criteria", "error", "asset_filename", "page_url"}
                }
                confidence = int(row.get("match_confidence", 0) or 0)
                await db.execute(
                    f"""
                    INSERT INTO {DB_SCHEMA}.audit_results (
                        session_id, asset_id, overall_verdict, adjudication_result,
                        criteria_checks, rationale, confidence_band
                    ) VALUES ($1, $2, $3, $4, $5, $6, $7)
                    """,
                    session_id,
                    row.get("asset_id"),
                    verdict,
                    bool(row.get("matches_criteria", False)),
                    json.dumps(checks, default=json_default),
                    row.get("match_rationale", "Completed"),
                    "high" if confidence >= 95 else ("borderline" if confidence >= 70 else "below_threshold"),
                )
    print("Audit results saved to the existing audit_results table.")


async def generate_ai_audit_summary(results_df: pd.DataFrame, audit_config: Dict[str, Any]) -> str:
    """Generate an optional summary from aggregates plus a bounded 40-row sample."""
    if results_df is None or results_df.empty:
        return "No audit results available."
    total = len(results_df)
    matches = int(results_df.get("matches_criteria", pd.Series(dtype=bool)).fillna(False).astype(bool).sum())
    errors = int(results_df.get("error", pd.Series(dtype=object)).notna().sum()) if "error" in results_df else 0
    safe_columns = [
        column for column in ("asset_id", "page_url", "matches_criteria", "match_confidence", "match_rationale", "gemini_description")
        if column in results_df.columns
    ]
    sample = results_df[safe_columns].head(40).where(pd.notna(results_df[safe_columns].head(40)), None).to_dict("records")
    prompt = f"""
Write a concise calibration summary. Do not claim perfect accuracy or success.
Audit goal: {audit_config.get('audit_goal', '')}
Audited canonical/occurrence rows: {total}
Rows marked as matches: {matches}
Evaluation errors: {errors}
Bounded evidence sample: {json.dumps(sample, default=str)}
"""

    def call_model():
        return _call_with_retry(lambda: client.models.generate_content(
            model=GEMINI_ORCHESTRATOR_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0),
        )).text

    return await asyncio.to_thread(call_model)

## End-to-End Execution and Telemetry

The execution layer coordinates retrieval, recall segmentation, deterministic reranking, bounded local visual verification, and final Gemini visual adjudication. Quick Mode is enforced after recall rescue and sends exactly up to 30 high-confidence plus up to 30 borderline candidates.


In [ ]:
# 5. Order-safe Stage 1 and baseline E2E helpers
import asyncio
import base64
import json
import os
from typing import Dict, List, Optional

import pandas as pd
from google.cloud import storage


def get_gcs_image_base64(path: str) -> str:
    """Return a bounded preview data URI; intended only for a small notebook preview."""
    try:
        payload = _download_bounded(path)
        preview = process_transparency(payload)
        if len(preview) > 4 * 1024 * 1024:
            return ""
        return "data:image/png;base64," + base64.b64encode(preview).decode("ascii")
    except Exception:
        return ""


async def _load_bounded_tag_vocabulary(*_args, **_kwargs) -> List[str]:
    """Compatibility hook: production deliberately avoids a database-wide tag scan.

    The table can contain tens of thousands of distinct labels. Sending that catalogue
    to Gemini, or rebuilding it by unnesting millions of rows per request, is both noisy
    and expensive. Retrieval instead uses a compact canonical vocabulary plus the audit's
    query terms directly against each asset's existing ``vision_tags`` array.
    """
    return []


async def generate_audit_config_only(user_goal: str, reference_image_path: Optional[str] = None) -> Dict:
    """Stage 1: visually describe an optional reference and return validated configuration JSON."""
    reference_description = None
    if reference_image_path:
        mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            image_part = types.Part.from_uri(file_uri=reference_image_path, mime_type=mime)
        else:
            payload = await asyncio.to_thread(_download_bounded, reference_image_path)
            image_part = types.Part.from_bytes(data=payload, mime_type=mime)
        forensic_prompt = """
Describe only visible evidence needed to find this target again: target identity if legible, standalone-versus-composite role, wordmarks/text, geometry, typography, solid-versus-gradient treatment, colors, crop/blur/occlusion, and the target's location in a larger canvas. Do not infer whether a brand design is current or legacy from outside knowledge. Do not invent cropped-out details.
"""

        def call_forensic():
            return _call_with_retry(lambda: client.models.generate_content(
                model=GEMINI_ORCHESTRATOR_MODEL,
                contents=[image_part, forensic_prompt],
                config=types.GenerateContentConfig(temperature=0.0),
            )).text

        reference_description = await asyncio.to_thread(call_forensic)
        print("Reference-image evidence extracted.")

    # Keep the model's output contract small and deterministic. The retrieval stage
    # expands these canonical labels with the audit keywords without enumerating the
    # approximately 70k distinct labels stored across the asset table.
    config = await asyncio.to_thread(
        generate_audit_config,
        user_goal,
        reference_description,
        WEB_AUDIT_VISION_TAGS,
    )
    print("Validated audit configuration generated.")
    return config


async def run_full_test_bench_pipeline_execution(
    audit_config: Dict,
    reference_image_path: Optional[str] = None,
    quick_mode: bool = False,
) -> pd.DataFrame:
    """Baseline E2E runner; the next production cell replaces it with recall safeguards."""
    search_results = await run_hybrid_search({}, audit_config, reference_image_path)
    if not search_results:
        return pd.DataFrame()
    high, edge, _ = detect_dropoff_flawless(pd.DataFrame(search_results))
    if quick_mode:
        high, edge = high.head(30), edge.head(30)
    return await run_llm_inference_on_dropoff_results(
        high, edge, audit_config, reference_image_path=reference_image_path
    )


async def run_full_test_bench_pipeline(
    user_goal: str,
    reference_image_path: Optional[str] = None,
    quick_mode: bool = False,
) -> pd.DataFrame:
    config = await generate_audit_config_only(user_goal, reference_image_path)
    return await run_full_test_bench_pipeline_execution(config, reference_image_path, quick_mode)

## Production Recall, Precision, and Memory-Safe Runtime

This mandatory, readable cell installs the retrieval and visual-audit implementation used below. It contains no compressed source, runtime decompression, external code links, database schema changes, or hidden uploads.


In [ ]:
# 6. Production Recall, Precision, and Memory-Safe Runtime
"""Readable production hardening for the Visual Asset Auditing notebook.

This module is intentionally database-schema neutral.  It reads only the
existing visual_assets columns used by the notebook and never issues DDL/DML.
"""

from __future__ import annotations

import asyncio
import hashlib
import io
import mimetypes
import re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageOps


def install_recall_precision_overrides(ns: Dict[str, Any]) -> None:
    """Install code-only, recall-first replacements into a running notebook.

    ``ns`` must be the notebook's ``globals()``.  The replacements deliberately
    preserve the existing database contract: ``visual_assets`` remains read-only.
    """

    original_dropoff = ns["detect_dropoff_flawless"]
    original_generate_config = ns["generate_audit_config_only"]
    original_llm_inference = ns["run_llm_inference_on_dropoff_results"]

    def get_image_mime_type(path: str) -> str:
        mime, _ = mimetypes.guess_type(path)
        return mime if mime and mime.startswith("image/") else "image/jpeg"

    def process_transparency(image_bytes: bytes, default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
        """Always return a real PNG, so bytes and declared MIME type agree."""
        try:
            with Image.open(io.BytesIO(image_bytes)) as source:
                # Do this header-only dimension check before decoding a source image.
                # 64 MP bounds even a worst-case RGBA decode on a 3 GB instance.
                if source.width * source.height > 64_000_000:
                    raise ValueError("Image exceeds the 64 MP App Engine processing safety limit")
                source = ImageOps.exif_transpose(source)
                # Bound active visual payloads.  1600 px retains detail while avoiding
                # sending multi-megabyte 4K/8K assets to every concurrent worker.
                max_side = 1600
                # JPEG draft mode avoids materialising a large source before resizing.
                try:
                    source.draft("RGB", (max_side, max_side))
                except Exception:
                    pass
                if max(source.size) > max_side:
                    resampling = getattr(Image, "Resampling", Image).LANCZOS
                    source.thumbnail((max_side, max_side), resampling)
                if source.mode in ("RGBA", "LA") or (source.mode == "P" and "transparency" in source.info):
                    rgba = source.convert("RGBA")
                    background = Image.new("RGBA", rgba.size, default_bg + (255,))
                    image = Image.alpha_composite(background, rgba).convert("RGB")
                else:
                    image = source.convert("RGB")
                output = io.BytesIO()
                image.save(output, format="PNG", optimize=True)
                return output.getvalue()
        except Image.DecompressionBombError as exc:
            raise ValueError(f"Image rejected by the decompression-bomb safety limit: {exc}") from exc
        except ValueError:
            raise
        except Exception as exc:
            raise ValueError(f"Image could not be decoded and normalized as PNG: {exc}") from exc

    def _goal_text(audit_config: Dict[str, Any]) -> str:
        values: Iterable[Any] = (
            audit_config.get("audit_goal", ""),
            audit_config.get("image_description", ""),
            " ".join(audit_config.get("inclusion_criteria", []) or []),
            " ".join(audit_config.get("exclusion_criteria", []) or []),
        )
        return " ".join(str(value) for value in values if value).lower()

    def _policy(audit_config: Dict[str, Any], has_reference: bool) -> Dict[str, Any]:
        text = _goal_text(audit_config)
        exact_terms = ("exact", "same image", "this image", "identical", "profile picture", "favicon")
        version_terms = ("legacy", "old", "outdated", "new", "current", "gradient", "solid", "version", "variant")
        exact_mode = has_reference and any(term in text for term in exact_terms)
        version_mode = any(term in text for term in version_terms)
        color_terms = ("gradient", "solid", "colour", "color", "red", "blue", "green", "yellow", "monochrome", "white", "black")
        color_is_discriminating = exact_mode or version_mode or any(term in text for term in color_terms)
        # Default to the declared App Engine 3 GB memory profile.  These clamp active
        # image payloads; they do not lower retrieval recall or change the database.
        runtime_profile = str(audit_config.get("runtime_profile", "app_engine_3gb")).strip().lower()
        constrained_runtime = runtime_profile in {"app_engine_3gb", "memory_constrained", "3gb"}
        llm_worker_cap = 2 if constrained_runtime else 4
        local_worker_cap = 2 if constrained_runtime else 4
        llm_workers = min(llm_worker_cap, max(1, int(audit_config.get("llm_concurrency", 2 if constrained_runtime else 4))))
        local_workers = min(local_worker_cap, max(1, int(audit_config.get("local_verification_concurrency", 2 if constrained_runtime else 4))))
        llm_batch_cap = 4 if constrained_runtime else 8
        llm_batch_size = min(llm_batch_cap, max(llm_workers, int(audit_config.get("llm_batch_size", llm_workers))))
        return {
            "mode": "exact_or_near_duplicate" if exact_mode else ("version_sensitive" if version_mode else "semantic_visual"),
            "color_is_discriminating": color_is_discriminating,
            "runtime_profile": runtime_profile,
            # The user may override these values in the Stage 2 cell.  They are
            # unique canonical candidates, not duplicate page occurrences.
            "retrieval_budget": min(50_000, max(1_000, int(audit_config.get("retrieval_budget", 30_000)))),
            "text_budget": min(10_000, max(0, int(audit_config.get("text_budget", 6_000)))),
            "tag_budget": min(10_000, max(0, int(audit_config.get("tag_budget", 6_000)))),
            "visual_audit_budget": min(1_000, max(60, int(audit_config.get("visual_audit_budget", 750 if exact_mode else 500)))),
            # These bound live image payloads, not recall.  Candidate work is batched.
            "llm_concurrency": llm_workers,
            "llm_batch_size": llm_batch_size,
            "local_verification_budget": max(0, int(audit_config.get("local_verification_budget", 200 if has_reference and (exact_mode or version_mode) else 0))),
            # A small, bounded probe from the low-score reserve is only used for
            # reference-image audits.  It can rescue a target embedded in a large canvas
            # before Quick Mode applies its fixed 30 + 30 quota.
            "local_rescue_budget": max(0, int(audit_config.get("local_rescue_budget", 120 if has_reference and (exact_mode or version_mode) else 0))),
            "local_rescue_min_score": float(audit_config.get("local_rescue_min_score", 0.48)),
            "local_verification_concurrency": local_workers,
        }

    def build_fused_query_text(context_dict: Dict[str, Any], is_negative: bool = False) -> str:
        """Create a query without erasing color/style evidence by default."""
        if is_negative:
            return "Exclude images that: " + "; ".join(context_dict.get("exclusion_criteria", []) or [])

        policy = context_dict.get("_search_policy", {})
        parts = [
            f"Audit goal: {context_dict.get('audit_goal', '')}",
            "Required visual signatures: " + "; ".join(context_dict.get("inclusion_criteria", []) or []),
        ]
        if context_dict.get("image_description"):
            parts.append(f"Reference-image analysis: {context_dict['image_description']}")
        if policy.get("color_is_discriminating"):
            parts.append("Color rendition, solid-versus-gradient treatment, and visual style are match determinants.")
        else:
            parts.append("Color is secondary unless it conflicts with an explicit visual criterion.")
        return " | ".join(part for part in parts if part)

    def _read_reference_bytes(path: Optional[str]) -> Optional[bytes]:
        if not path or path.startswith("gs://"):
            return None
        try:
            with open(path, "rb") as source:
                return source.read()
        except OSError:
            return None

    async def generate_audit_config_only(user_goal: str, reference_image_path: Optional[str] = None) -> Dict[str, Any]:
        config = await original_generate_config(user_goal, reference_image_path)
        policy = _policy(config, bool(reference_image_path))
        config["_search_policy"] = policy
        # This additional instruction counteracts the old notebook's global
        # color-independence rule only when exact/version matching needs it.
        if policy["color_is_discriminating"]:
            color_criterion = (
                "The candidate must preserve the required color treatment and visual rendition "
                "of the target, including solid-versus-gradient styling when visible."
            )
            criteria = list(config.get("inclusion_criteria", []) or [])
            if not any("gradient" in str(item).lower() or "color treatment" in str(item).lower() for item in criteria):
                criteria.append(color_criterion)
            config["inclusion_criteria"] = criteria
            config["audit_instructions"] = (
                config.get("audit_instructions", "")
                + "\nFor this audit, color rendition and gradient-versus-solid treatment are evidence. "
                "Do not pass a visually similar newer or alternate rendition when those features differ."
            )
            config["adjudication_logic"] = (
                config.get("adjudication_logic", "")
                + " IF the visible color treatment or solid-versus-gradient rendition conflicts with the target, "
                "THEN fail; ELSE continue the remaining visual checks."
            )
        print(
            "Recall/precision policy: "
            f"{policy['mode']} | retrieval budget={policy['retrieval_budget']:,} unique candidates | "
            f"visual audit budget={policy['visual_audit_budget']:,}"
        )
        return config

    async def embed_audit_context(context_dict: Dict[str, Any], reference_image_path: Optional[str] = None) -> Tuple[List[float], List[float]]:
        """Embed the actual reference image plus grounded query text.

        The stored column is still queried exactly as-is; no embedding data is
        written back to AlloyDB.
        """
        types = ns["types"]
        client = ns["client"]
        embedding_model = ns["EMBEDDING_MODEL"]
        pos_contents: List[Any] = []
        if reference_image_path:
            mime = get_image_mime_type(reference_image_path)
            if reference_image_path.startswith("gs://"):
                pos_contents.append(types.Part.from_uri(file_uri=reference_image_path, mime_type=mime))
            else:
                payload = _read_reference_bytes(reference_image_path)
                if payload:
                    pos_contents.append(types.Part.from_bytes(data=payload, mime_type=mime))
        pos_contents.append(build_fused_query_text(context_dict)[:3_000])
        neg_contents = [build_fused_query_text(context_dict, is_negative=True)[:1_500]]

        def _embed(contents: List[Any]) -> List[float]:
            response = client.models.embed_content(
                model=embedding_model,
                contents=contents,
                config=types.EmbedContentConfig(output_dimensionality=768),
            )
            return response.embeddings[0].values

        loop = asyncio.get_running_loop()
        pos_vec, neg_vec = await asyncio.gather(
            loop.run_in_executor(None, _embed, pos_contents),
            loop.run_in_executor(None, _embed, neg_contents),
        )
        return pos_vec, neg_vec

    def _bounded_tag_terms(audit_context: Dict[str, Any], maximum: int = 24) -> List[str]:
        """Build a small, relevant tag query without loading the 70k-label catalogue."""
        configured = [str(value).strip() for value in audit_context.get("vision_tag_filter", []) or []]
        keywords = [str(value).strip() for value in audit_context.get("search_keywords", []) or []]
        goal = _goal_text(audit_context)
        canonical = list(configured)
        routing = {
            "favicon": ("Icon", "Logo", "Symbol", "Graphic design"),
            "logo": ("Logo", "Brand", "Symbol", "Product"),
            "profile": ("Portrait", "Person", "Face", "Photography"),
            "portrait": ("Portrait", "Person", "Face", "Photography"),
            "person": ("Person", "Face", "Portrait"),
            "screenshot": ("Screenshot", "Web page", "Computer", "Technology"),
            "button": ("Button", "Text", "Font", "Graphic design"),
            "banner": ("Banner", "Advertising", "Graphic design", "Text"),
        }
        for trigger, labels in routing.items():
            if trigger in goal:
                canonical.extend(labels)
        # Exact array overlap remains index-friendly when a GIN array index already exists.
        # Include title-cased single-token query terms for custom labels, but keep the list bounded.
        canonical.extend(term.title() for term in keywords if re.fullmatch(r"[A-Za-z][A-Za-z0-9_-]{1,39}", term))
        output: List[str] = []
        seen = set()
        for term in canonical:
            cleaned = re.sub(r"\s+", " ", term).strip()
            key = cleaned.casefold()
            if cleaned and key not in seen:
                seen.add(key)
                output.append(cleaned)
            if len(output) >= maximum:
                break
        return output

    async def run_hybrid_search(
        scope_config: Dict[str, Any],
        audit_context: Dict[str, Any],
        reference_image_path: Optional[str] = None,
        limit: int = 10_000,
    ) -> List[Dict[str, Any]]:
        """Recall-first hybrid retrieval with one direct ANN lane per modality.

        It avoids the previous compound ``ORDER BY`` expression, which made
        index usage and ANN recall difficult to reason about.  It also ranks
        FTS/tag results before RRF, and keeps source evidence for every item.
        """
        register_vector = ns["register_vector"]
        engine, _ = await ns["get_alloydb_connection"]()
        pos_vec, neg_vec = await embed_audit_context(audit_context, reference_image_path)
        policy = audit_context.setdefault("_search_policy", _policy(audit_context, bool(reference_image_path)))
        vector_limit = max(limit, policy["retrieval_budget"])
        vector_raw_limit = min(50_000, max(vector_limit, vector_limit * 2))
        text_limit = policy["text_budget"]
        text_raw_limit = min(20_000, max(text_limit, text_limit * 2))
        tag_limit = policy["tag_budget"]
        tag_raw_limit = min(20_000, max(tag_limit, tag_limit * 2))
        keywords = [str(term).strip() for term in audit_context.get("search_keywords", []) or [] if str(term).strip()]
        tags = _bounded_tag_terms(audit_context)
        query_timeout = ns.get("DB_QUERY_TIMEOUT_SECONDS", 90)
        reference_bytes = _read_reference_bytes(reference_image_path)
        reference_sha256 = hashlib.sha256(reference_bytes).hexdigest() if reference_bytes else None
        select_columns = "asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash"

        async def vector_lane() -> List[Dict[str, Any]]:
            async with engine.connect() as conn:
                raw = await conn.get_raw_connection()
                db = raw.driver_connection
                await register_vector(db)
                rows = await db.fetch(
                    f"""
                    WITH nearest AS MATERIALIZED (
                        SELECT {select_columns},
                               (embedding <=> $1::vector) AS vector_distance,
                               (embedding <=> $2::vector) AS negative_distance
                        FROM {ns['DB_SCHEMA']}.visual_assets
                        ORDER BY embedding <=> $1::vector
                        LIMIT $3
                    ), canonical AS (
                        SELECT DISTINCT ON (COALESCE(content_hash::text, asset_id::text)) *
                        FROM nearest
                        ORDER BY COALESCE(content_hash::text, asset_id::text), vector_distance, asset_id
                    )
                    SELECT * FROM canonical
                    ORDER BY vector_distance, asset_id
                    LIMIT $4
                    """,
                    pos_vec,
                    neg_vec,
                    vector_raw_limit,
                    vector_limit,
                    timeout=query_timeout,
                )
                return [dict(row) for row in rows]

        async def fts_lane() -> List[Dict[str, Any]]:
            if not keywords:
                return []
            query = " OR ".join(keywords)
            async with engine.connect() as conn:
                raw = await conn.get_raw_connection()
                db = raw.driver_connection
                rows = await db.fetch(
                    f"""
                    WITH ranked AS MATERIALIZED (
                        SELECT {select_columns},
                               ts_rank_cd(
                                   to_tsvector('english', gemini_description),
                                   websearch_to_tsquery('english', $1)
                               ) AS text_rank
                        FROM {ns['DB_SCHEMA']}.visual_assets
                        WHERE to_tsvector('english', gemini_description)
                              @@ websearch_to_tsquery('english', $1)
                        ORDER BY text_rank DESC, asset_id
                        LIMIT $2
                    ), canonical AS (
                        SELECT DISTINCT ON (COALESCE(content_hash::text, asset_id::text)) *
                        FROM ranked
                        ORDER BY COALESCE(content_hash::text, asset_id::text), text_rank DESC, asset_id
                    )
                    SELECT * FROM canonical
                    ORDER BY text_rank DESC, asset_id
                    LIMIT $3
                    """,
                    query,
                    text_raw_limit,
                    text_limit,
                    timeout=query_timeout,
                )
                return [dict(row) for row in rows]

        async def tag_lane() -> List[Dict[str, Any]]:
            if not tags:
                return []
            async with engine.connect() as conn:
                raw = await conn.get_raw_connection()
                db = raw.driver_connection
                rows = await db.fetch(
                    f"""
                    WITH ranked AS MATERIALIZED (
                        SELECT {select_columns},
                               (
                                   SELECT COUNT(*)
                                   FROM unnest(COALESCE(vision_tags, ARRAY[]::text[])) AS asset_tag
                                   WHERE asset_tag = ANY($1::text[])
                               ) AS tag_hits
                        FROM {ns['DB_SCHEMA']}.visual_assets
                        WHERE vision_tags && $1::text[]
                        ORDER BY tag_hits DESC, asset_id
                        LIMIT $2
                    ), canonical AS (
                        SELECT DISTINCT ON (COALESCE(content_hash::text, asset_id::text)) *
                        FROM ranked
                        ORDER BY COALESCE(content_hash::text, asset_id::text), tag_hits DESC, asset_id
                    )
                    SELECT * FROM canonical
                    ORDER BY tag_hits DESC, asset_id
                    LIMIT $3
                    """,
                    tags,
                    tag_raw_limit,
                    tag_limit,
                    timeout=query_timeout,
                )
                return [dict(row) for row in rows]

        async def exact_hash_lane() -> List[Dict[str, Any]]:
            if not reference_sha256:
                return []
            async with engine.connect() as conn:
                raw = await conn.get_raw_connection()
                db = raw.driver_connection
                rows = await db.fetch(
                    f"""
                    SELECT DISTINCT ON (content_hash) {select_columns}
                    FROM {ns['DB_SCHEMA']}.visual_assets
                    WHERE content_hash = $1
                    ORDER BY content_hash, asset_id
                    """,
                    reference_sha256,
                    timeout=query_timeout,
                )
                return [dict(row) for row in rows]

        vector_rows, text_rows, tag_rows, exact_rows = await asyncio.gather(
            vector_lane(), fts_lane(), tag_lane(), exact_hash_lane()
        )

        # Weighted RRF preserves all independent evidence.  Unlike the old
        # implementation, local metadata can nudge a score but cannot multiply
        # it by 8x and drown out visual similarity.
        weights = (("vector", vector_rows, 0.65), ("text", text_rows, 0.20), ("tag", tag_rows, 0.15), ("exact_hash", exact_rows, 4.0))
        fused: Dict[str, Dict[str, Any]] = {}
        rrf_k = 60
        for source, rows, weight in weights:
            for rank, row in enumerate(rows, start=1):
                canonical_key = str(row.get("content_hash") or row["asset_id"])
                item = fused.setdefault(
                    canonical_key,
                    {
                        "data": row,
                        "score": 0.0,
                        "sources": [],
                        "source_ranks": {},
                        "vector_distance": row.get("vector_distance"),
                        "negative_distance": row.get("negative_distance"),
                    },
                )
                item["score"] += weight / (rrf_k + rank)
                item["sources"].append(source)
                item["source_ranks"][source] = min(rank, item["source_ranks"].get(source, rank))
                if row.get("vector_distance") is not None:
                    item["vector_distance"] = row["vector_distance"]
                    item["negative_distance"] = row.get("negative_distance")

        output: List[Dict[str, Any]] = []
        for canonical_key, item in fused.items():
            row = item["data"]
            if "exact_hash" in item["sources"]:
                item["score"] += 0.75
            positive_distance = item.get("vector_distance")
            negative_distance = item.get("negative_distance")
            if positive_distance is not None and negative_distance is not None:
                # Positive closer than negative => positive margin. Keep this bounded so RRF remains primary.
                item["score"] += 0.003 * float(np.clip(negative_distance - positive_distance, -1.0, 1.0))
            output.append(
                {
                    **row,
                    "canonical_key": canonical_key,
                    "relevance_score": float(item["score"]),
                    "retrieval_sources": sorted(set(item["sources"])),
                    "source_ranks": item["source_ranks"],
                    "promoted_by_keyword_or_tag": bool({"text", "tag"} & set(item["sources"])),
                    "exact_content_hash_match": "exact_hash" in item["sources"],
                    "vector_distance": positive_distance if positive_distance is not None else 1.0,
                    "negative_distance": negative_distance,
                }
            )
        output.sort(key=lambda row: row["relevance_score"], reverse=True)
        print(
            "Retrieval lanes — "
            f"vector: {len(vector_rows):,}, text: {len(text_rows):,}, tags: {len(tag_rows):,}, "
            f"exact-hash: {len(exact_rows):,}, unique candidates: {len(output):,}"
        )
        return output

    def detect_dropoff_flawless(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Keep the existing segmentation labels, but never treat Low as deleted."""
        if df is None or df.empty:
            return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
        if len(df) < 4:
            high = df.sort_values("relevance_score", ascending=False).copy()
            high["segmentation_band"] = "high"
            return high, pd.DataFrame(columns=high.columns), pd.DataFrame(columns=high.columns)
        high, edge, low = original_dropoff(df, sensitivity)
        for frame, label in ((high, "high"), (edge, "borderline"), (low, "recall_reserve")):
            if not frame.empty:
                frame["segmentation_band"] = label
        return high, edge, low

    def _visual_candidate_set(high: pd.DataFrame, edge: pd.DataFrame, low: pd.DataFrame, config: Dict[str, Any], quick_mode: bool) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Select final visual-audit candidates without silently discarding a band.

        Quick Mode is deliberately strict: up to 30 candidates from High and up to 30
        candidates from Borderline are sent to the vision judge.  A low-score image can
        enter Borderline only through the bounded local-reference rescue pass below,
        never because the quota pools all bands together.
        """
        policy = config["_search_policy"]
        high = high.sort_values("relevance_score", ascending=False).copy() if not high.empty else high
        edge = edge.copy()
        if not edge.empty:
            rescue_scores = edge["local_rescue_score"] if "local_rescue_score" in edge.columns else pd.Series(0.0, index=edge.index)
            edge["_rescue_order"] = pd.to_numeric(rescue_scores, errors="coerce").fillna(0.0)
            edge = edge.sort_values(["_rescue_order", "relevance_score"], ascending=False).drop(columns=["_rescue_order"])
        if quick_mode:
            return high.head(30).reset_index(drop=True), edge.head(30).reset_index(drop=True)

        all_candidates = pd.concat([high, edge, low], ignore_index=True)
        if all_candidates.empty:
            return pd.DataFrame(), pd.DataFrame()
        all_candidates = all_candidates.drop_duplicates(subset=["canonical_key"], keep="first")
        exact = all_candidates[all_candidates.get("exact_content_hash_match", False) == True]
        lane_rows = []
        for lane in ("vector", "text", "tag"):
            lane_rows.append(all_candidates[all_candidates["retrieval_sources"].apply(lambda sources: lane in sources)].head(100))
        protected = pd.concat([exact, *lane_rows], ignore_index=True).drop_duplicates(subset=["canonical_key"], keep="first")
        budget = policy["visual_audit_budget"]
        fused_head = all_candidates.head(max(budget, len(protected)))
        selected = pd.concat([protected, fused_head], ignore_index=True).drop_duplicates(subset=["canonical_key"], keep="first")
        selected = selected.sort_values("relevance_score", ascending=False).head(budget).reset_index(drop=True)
        return (
            selected[selected["segmentation_band"] == "high"].copy(),
            selected[selected["segmentation_band"] != "high"].copy(),
        )

    async def run_semantic_reranking_and_filter(
        df_high: pd.DataFrame,
        df_edge: pd.DataFrame,
        audit_context: Dict[str, Any],
        max_workers: int = 0,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Compatibility shim: keep the score bands but make no remote LLM calls.

        The legacy reranker only evaluates descriptions, tags, and filenames.  It is
        intentionally disabled because it adds latency without seeing candidate pixels.
        """
        print("Semantic LLM reranking disabled: retaining deterministic retrieval ranks.")
        high = df_high.sort_values("relevance_score", ascending=False).reset_index(drop=True) if not df_high.empty else df_high
        edge = df_edge.sort_values("relevance_score", ascending=False).reset_index(drop=True) if not df_edge.empty else df_edge
        return high, edge, pd.DataFrame(columns=(high.columns if not high.empty else edge.columns))

    def _download_asset_bytes(path: Optional[str]) -> Optional[bytes]:
        """Read one asset from its existing GCS/local path; no database access occurs."""
        if not path:
            return None
        try:
            if path.startswith("gs://"):
                bucket_name, blob_name = path[5:].split("/", 1)
                storage_client = ns.get("_audit_storage_client")
                if storage_client is None:
                    storage_client = ns["storage"].Client(project=ns.get("PROJECT_ID"))
                    ns["_audit_storage_client"] = storage_client
                payload = storage_client.bucket(bucket_name).blob(blob_name).download_as_bytes(
                    start=0, end=ns.get("MAX_SOURCE_BYTES", 48 * 1024 * 1024),
                    timeout=ns.get("GCS_DOWNLOAD_TIMEOUT_SECONDS", 30),
                )
                if len(payload) > ns.get("MAX_SOURCE_BYTES", 48 * 1024 * 1024):
                    return None
                return payload
            with open(path, "rb") as asset_file:
                payload = asset_file.read(ns.get("MAX_SOURCE_BYTES", 48 * 1024 * 1024) + 1)
            return payload if len(payload) <= ns.get("MAX_SOURCE_BYTES", 48 * 1024 * 1024) else None
        except Exception:
            return None

    def _local_visual_score(reference_rgb: np.ndarray, candidate_bytes: Optional[bytes]) -> Dict[str, Any]:
        """CPU-only duplicate/crop evidence for a small, already-shortlisted set.

        ORB supports partial/cropped copies; multi-scale template matching supports
        simple logos; dHash helps with same-image or resized near-duplicates.  A score
        only boosts ordering and never filters a candidate, protecting recall.
        """
        if not candidate_bytes:
            return {"local_visual_score": 0.0, "local_visual_method": "unavailable"}
        try:
            import cv2
            # OpenCV stays CPU-only and cannot create per-worker native thread pools.
            try:
                cv2.setNumThreads(1)
            except Exception:
                pass

            with Image.open(io.BytesIO(candidate_bytes)) as candidate_source:
                if candidate_source.width * candidate_source.height > 64_000_000:
                    raise ValueError("Candidate exceeds the 64 MP App Engine processing safety limit")
                candidate_source = ImageOps.exif_transpose(candidate_source)
                # Preserve small embedded targets while bounding App Engine image work.
                try:
                    candidate_source.draft("RGB", (2048, 2048))
                except Exception:
                    pass
                candidate_source.thumbnail((2048, 2048), getattr(Image, "Resampling", Image).LANCZOS)
                candidate_rgb = np.asarray(candidate_source.convert("RGB"))
            reference_gray = cv2.cvtColor(reference_rgb, cv2.COLOR_RGB2GRAY)
            candidate_gray = cv2.cvtColor(candidate_rgb, cv2.COLOR_RGB2GRAY)
            methods: List[str] = []
            score = 0.0

            # dHash: high confidence for same/resized images.  Do not let it dominate
            # when aspect ratios disagree, because a small embedded target is expected.
            reference_ratio = reference_gray.shape[1] / max(reference_gray.shape[0], 1)
            candidate_ratio = candidate_gray.shape[1] / max(candidate_gray.shape[0], 1)
            if abs(reference_ratio - candidate_ratio) <= 0.15:
                def _dhash(image: np.ndarray) -> np.ndarray:
                    resized = cv2.resize(image, (9, 8), interpolation=cv2.INTER_AREA)
                    return resized[:, 1:] > resized[:, :-1]

                hash_distance = np.count_nonzero(_dhash(reference_gray) != _dhash(candidate_gray))
                hash_score = 1.0 - (hash_distance / 64.0)
                if hash_score >= 0.72:
                    score = max(score, float(hash_score))
                    methods.append("dhash")

            # Multi-scale template matching is particularly useful for small, simple
            # marks that do not yield many ORB keypoints.
            candidate_height, candidate_width = candidate_gray.shape[:2]
            reference_height, reference_width = reference_gray.shape[:2]
            template_score = 0.0
            for scale in (0.03, 0.05, 0.08, 0.12, 0.18, 0.28, 0.42, 0.60, 0.80, 1.00):
                width = max(16, int(reference_width * scale))
                height = max(16, int(reference_height * scale))
                if width > candidate_width or height > candidate_height:
                    continue
                template = cv2.resize(reference_gray, (width, height), interpolation=cv2.INTER_AREA)
                _, current, _, _ = cv2.minMaxLoc(cv2.matchTemplate(candidate_gray, template, cv2.TM_CCOEFF_NORMED))
                template_score = max(template_score, float(current))
            if template_score >= 0.45:
                score = max(score, template_score)
                methods.append("template")

            # ORB + RANSAC verifies geometric correspondence across crop/scale changes.
            orb = cv2.ORB_create(nfeatures=1_000, fastThreshold=7)
            keypoints_ref, descriptors_ref = orb.detectAndCompute(reference_gray, None)
            keypoints_candidate, descriptors_candidate = orb.detectAndCompute(candidate_gray, None)
            if descriptors_ref is not None and descriptors_candidate is not None:
                matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
                pairs = matcher.knnMatch(descriptors_ref, descriptors_candidate, k=2)
                good = [pair[0] for pair in pairs if len(pair) == 2 and pair[0].distance < 0.75 * pair[1].distance]
                if len(good) >= 4:
                    source_points = np.float32([keypoints_ref[match.queryIdx].pt for match in good]).reshape(-1, 1, 2)
                    target_points = np.float32([keypoints_candidate[match.trainIdx].pt for match in good]).reshape(-1, 1, 2)
                    _, mask = cv2.findHomography(source_points, target_points, cv2.RANSAC, 5.0)
                    inliers = int(mask.ravel().sum()) if mask is not None else 0
                    feature_score = min(1.0, 0.35 + 0.05 * inliers + 0.01 * len(good))
                    if feature_score >= 0.50:
                        score = max(score, feature_score)
                        methods.append("orb_ransac")
            return {
                "local_visual_score": round(float(score), 4),
                "local_visual_method": "+".join(methods) if methods else "no_strong_local_signal",
            }
        except ImportError:
            return {"local_visual_score": 0.0, "local_visual_method": "opencv_unavailable"}
        except Exception:
            return {"local_visual_score": 0.0, "local_visual_method": "local_match_error"}

    async def _apply_local_reference_verification(
        df_high: pd.DataFrame,
        df_edge: pd.DataFrame,
        audit_config: Dict[str, Any],
        reference_image_path: Optional[str],
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Add bounded CPU-only image comparison evidence without excluding results."""
        policy = audit_config["_search_policy"]
        combined = pd.concat([df_high, df_edge], ignore_index=True)
        budget = min(len(combined), policy["local_verification_budget"])
        if not reference_image_path or not budget or combined.empty:
            return df_high, df_edge
        reference_bytes = _download_asset_bytes(reference_image_path)
        if not reference_bytes:
            print("Local reference verification skipped: reference image could not be read.")
            return df_high, df_edge
        try:
            with Image.open(io.BytesIO(reference_bytes)) as reference_source:
                reference_source = ImageOps.exif_transpose(reference_source).convert("RGB")
                reference_source.thumbnail((768, 768), getattr(Image, "Resampling", Image).LANCZOS)
                reference_rgb = np.asarray(reference_source)
        except Exception:
            print("Local reference verification skipped: unsupported reference image.")
            return df_high, df_edge

        combined = combined.copy()
        combined["local_visual_score"] = 0.0
        combined["local_visual_method"] = "not_checked"
        checked = combined.head(budget)
        workers = policy["local_verification_concurrency"]
        batch_size = max(workers, workers * 2)
        semaphore = asyncio.Semaphore(workers)
        loop = asyncio.get_running_loop()

        async def _score(index: int, row: pd.Series, executor: ThreadPoolExecutor) -> Tuple[int, Dict[str, Any]]:
            async with semaphore:
                payload = await loop.run_in_executor(executor, _download_asset_bytes, row.get("gcs_raw_path"))
                score = await loop.run_in_executor(executor, _local_visual_score, reference_rgb, payload)
                return index, score

        print(f"Local image verification: checking {len(checked):,} candidates with {workers} bounded workers (no LLM calls).")
        with ThreadPoolExecutor(max_workers=workers) as executor:
            for start in range(0, len(checked), batch_size):
                batch = checked.iloc[start:start + batch_size]
                updates = await asyncio.gather(*[_score(index, row, executor) for index, row in batch.iterrows()])
                for index, score in updates:
                    combined.loc[index, "local_visual_score"] = score["local_visual_score"]
                    combined.loc[index, "local_visual_method"] = score["local_visual_method"]

        # Preserve deterministic retrieval order as the primary signal.  Local evidence
        # is a bounded boost, never a rejection rule.
        rank_signal = 1.0 - (np.arange(len(combined)) / max(len(combined), 1))
        combined["local_rerank_score"] = rank_signal + 0.35 * combined["local_visual_score"].astype(float)
        combined = combined.sort_values(["local_rerank_score", "relevance_score"], ascending=False).reset_index(drop=True)
        local_high = combined[combined["segmentation_band"] == "high"].copy()
        local_edge = combined[combined["segmentation_band"] != "high"].copy()
        return local_high, local_edge

    async def _rescue_low_reference_candidates(
        high: pd.DataFrame,
        edge: pd.DataFrame,
        low: pd.DataFrame,
        audit_config: Dict[str, Any],
        reference_image_path: Optional[str],
        quick_mode: bool,
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Promote only strong local image evidence from a bounded low-score probe.

        Whole-image embeddings can underrate a small logo/profile image inside a large
        hero or screenshot.  This stage samples boundary, visual, and distributed low
        candidates, uses CPU dHash/template/ORB evidence, and promotes only strong
        hits to Borderline.  It never rejects or overwrites High candidates.
        """
        policy = audit_config["_search_policy"]
        if low.empty or not reference_image_path or not policy["local_rescue_budget"]:
            return high, edge
        reference_bytes = _download_asset_bytes(reference_image_path)
        if not reference_bytes:
            return high, edge
        try:
            with Image.open(io.BytesIO(reference_bytes)) as reference_source:
                reference_source = ImageOps.exif_transpose(reference_source).convert("RGB")
                reference_source.thumbnail((768, 768), getattr(Image, "Resampling", Image).LANCZOS)
                reference_rgb = np.asarray(reference_source)
        except Exception:
            return high, edge

        limit = min(len(low), policy["local_rescue_budget"], 72 if quick_mode else policy["local_rescue_budget"])
        ranked = low.sort_values("relevance_score", ascending=False).copy()
        probes = [ranked.head(max(1, limit // 2))]
        if "vector_distance" in ranked.columns:
            probes.append(ranked.sort_values("vector_distance", ascending=True, na_position="last").head(max(1, limit // 3)))
        for lane in ("text", "tag"):
            lane_rows = ranked[ranked["retrieval_sources"].apply(lambda sources: lane in (sources or []))]
            if not lane_rows.empty:
                probes.append(lane_rows.head(max(1, limit // 8)))
        if len(ranked) > 1:
            distributed = np.linspace(0, len(ranked) - 1, num=min(max(6, limit // 6), len(ranked))).astype(int)
            probes.append(ranked.iloc[distributed])
        probe_df = pd.concat(probes, ignore_index=True).drop_duplicates(subset=["canonical_key"], keep="first").head(limit)
        if probe_df.empty:
            return high, edge

        workers = policy["local_verification_concurrency"]
        loop = asyncio.get_running_loop()
        semaphore = asyncio.Semaphore(workers)

        async def _score(index: int, row: pd.Series, executor: ThreadPoolExecutor) -> Tuple[int, Dict[str, Any]]:
            async with semaphore:
                payload = await loop.run_in_executor(executor, _download_asset_bytes, row.get("gcs_raw_path"))
                evidence = await loop.run_in_executor(executor, _local_visual_score, reference_rgb, payload)
                return index, evidence

        print(f"Low-score recall rescue: probing {len(probe_df):,} candidates with {workers} bounded CPU workers.")
        scored = probe_df.copy()
        scored["local_rescue_score"] = 0.0
        scored["local_rescue_method"] = "not_checked"
        with ThreadPoolExecutor(max_workers=workers) as executor:
            for start in range(0, len(scored), max(workers, workers * 2)):
                batch = scored.iloc[start:start + max(workers, workers * 2)]
                for index, evidence in await asyncio.gather(*[_score(index, row, executor) for index, row in batch.iterrows()]):
                    scored.loc[index, "local_rescue_score"] = evidence["local_visual_score"]
                    scored.loc[index, "local_rescue_method"] = evidence["local_visual_method"]

        rescued = scored[scored["local_rescue_score"] >= policy["local_rescue_min_score"]].copy()
        if rescued.empty:
            return high, edge
        rescued["segmentation_band"] = "borderline"
        high_keys = set(high.get("canonical_key", pd.Series(dtype=str)).astype(str))
        edge_keys = set(edge.get("canonical_key", pd.Series(dtype=str)).astype(str))
        rescued = rescued[~rescued["canonical_key"].astype(str).isin(high_keys | edge_keys)]
        if rescued.empty:
            return high, edge
        expanded_edge = pd.concat([rescued, edge], ignore_index=True).drop_duplicates(subset=["canonical_key"], keep="first")
        expanded_edge = expanded_edge.sort_values(["local_rescue_score", "relevance_score"], ascending=False, na_position="last")
        print(f"Low-score recall rescue promoted {len(rescued):,} candidates to Borderline based on local visual evidence.")
        return high, expanded_edge

    async def run_llm_inference_on_dropoff_results(
        df_high: pd.DataFrame,
        df_edge: pd.DataFrame,
        audit_config: Dict[str, Any],
        max_workers: int = 8,
        reference_image_path: Optional[str] = None,
    ) -> pd.DataFrame:
        """Run the final visual judge in bounded batches; never queue all images at once."""
        candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
        if candidates_df.empty:
            return pd.DataFrame()
        single_audit = ns.get("run_llm_audit_single")
        if not callable(single_audit):
            return await original_llm_inference(df_high, df_edge, audit_config, max_workers=max_workers, reference_image_path=reference_image_path)

        policy = audit_config.setdefault("_search_policy", _policy(audit_config, bool(reference_image_path)))
        # Use the policy caps; direct runtime parameters may not bypass 3 GB safety.
        workers = policy["llm_concurrency"]
        batch_size = policy["llm_batch_size"]
        candidates = candidates_df.to_dict(orient="records")
        loop = asyncio.get_running_loop()
        semaphore = asyncio.Semaphore(workers)

        def _reference_part() -> Optional[Any]:
            payload = _download_asset_bytes(reference_image_path)
            if not payload:
                return None
            return ns["types"].Part.from_bytes(data=process_transparency(payload), mime_type="image/png")

        async def _audit_one(asset: Dict[str, Any], executor: ThreadPoolExecutor, reference_part: Optional[Any]) -> Dict[str, Any]:
            async with semaphore:
                return await single_audit(asset, audit_config, _executor=executor, reference_image_part=reference_part)

        print(f"Final visual audit: {len(candidates):,} candidates in batches of {batch_size} (max {workers} active image payloads).")
        results: List[Dict[str, Any]] = []
        with ThreadPoolExecutor(max_workers=workers) as executor:
            reference_part = await loop.run_in_executor(executor, _reference_part) if reference_image_path else None
            for start in range(0, len(candidates), batch_size):
                batch = candidates[start:start + batch_size]
                results.extend(await asyncio.gather(*[_audit_one(asset, executor, reference_part) for asset in batch]))

        results_df = pd.DataFrame(results)
        guardrail = ns.get("enforce_zero_false_positives_rules")
        if callable(guardrail) and not results_df.empty:
            results_df = guardrail(results_df)
        return results_df.sort_values("relevance_score", ascending=False, na_position="last").reset_index(drop=True)

    async def _expand_passing_occurrences(results_df: pd.DataFrame) -> pd.DataFrame:
        """Expand one audited canonical asset to every page occurrence in one read-only query."""
        if results_df.empty or "content_hash" not in results_df.columns:
            return results_df
        match_mask = results_df["matches_criteria"].fillna(False).astype(bool) if "matches_criteria" in results_df else pd.Series(False, index=results_df.index)
        matched = results_df[match_mask].copy()
        hashes = [str(value) for value in matched["content_hash"].dropna().unique() if str(value)]
        if not hashes:
            return results_df
        engine, _ = await ns["get_alloydb_connection"]()
        async with engine.connect() as conn:
            raw = await conn.get_raw_connection()
            db = raw.driver_connection
            rows = await db.fetch(
                f"""
                SELECT DISTINCT ON (content_hash, page_url)
                       asset_id, content_hash, page_url, gcs_raw_path, asset_filename,
                       format, vision_tags, gemini_description
                FROM {ns['DB_SCHEMA']}.visual_assets
                WHERE content_hash = ANY($1::text[])
                ORDER BY content_hash, page_url, asset_id
                """,
                hashes,
                timeout=ns.get("DB_QUERY_TIMEOUT_SECONDS", 90),
            )
        canonical_by_hash = {str(row["content_hash"]): row for _, row in matched.iterrows()}
        expanded: List[Dict[str, Any]] = []
        for occurrence in rows:
            occurrence_data = dict(occurrence)
            canonical = canonical_by_hash.get(str(occurrence_data["content_hash"]))
            if canonical is None:
                continue
            result = canonical.to_dict()
            result.update(occurrence_data)
            result["canonical_match_asset_id"] = canonical.get("asset_id")
            result["occurrence_expanded"] = True
            expanded.append(result)
        unmatched = results_df[~match_mask].to_dict("records")
        final_rows = [*expanded, *unmatched]
        final = pd.DataFrame(final_rows) if final_rows else results_df
        print(f"Occurrence expansion: {len(matched):,} canonical matches → {len(expanded):,} matching page occurrences.")
        return final

    async def run_full_test_bench_pipeline_execution(
        audit_config: Dict[str, Any],
        reference_image_path: Optional[str] = None,
        quick_mode: bool = False,
    ) -> pd.DataFrame:
        """Recall-first E2E runner with no database writes or schema changes."""
        audit_config.setdefault("_search_policy", _policy(audit_config, bool(reference_image_path)))
        print("\n2. Running recall-first hybrid search…")
        search_results = await run_hybrid_search({}, audit_config, reference_image_path)
        if not search_results:
            return pd.DataFrame()
        candidates = pd.DataFrame(search_results)
        policy = audit_config["_search_policy"]
        exact_fast = pd.DataFrame()
        if policy["mode"] == "exact_or_near_duplicate" and "exact_content_hash_match" in candidates:
            exact_fast = candidates[candidates["exact_content_hash_match"].fillna(False)].copy()
            if not exact_fast.empty:
                exact_fast["matches_criteria"] = True
                exact_fast["match_confidence"] = 100
                exact_fast["match_rationale"] = "Cryptographic SHA-256 content-hash identity with the uploaded reference; visual LLM call bypassed."
                exact_fast["visual_analysis_step_by_step"] = "Exact stored-byte identity established by SHA-256."
                exact_fast["adjudication_method"] = "sha256_exact"
                candidates = candidates[~candidates["exact_content_hash_match"].fillna(False)].copy()
                print(f"Exact-hash fast path: {len(exact_fast):,} canonical candidate(s) bypass visual inference.")
        print("\n3. Labelling score bands (no candidates are discarded at this step)…")
        high, edge, low = detect_dropoff_flawless(candidates)
        high, edge = await _rescue_low_reference_candidates(
            high, edge, low, audit_config, reference_image_path, quick_mode
        )
        visual_high, visual_edge = _visual_candidate_set(high, edge, low, audit_config, quick_mode)
        print(
            f"Visual audit set: {len(visual_high) + len(visual_edge):,} canonical candidates "
            f"from {len(candidates):,} retrieved unique candidates."
        )
        visual_high, visual_edge = await _apply_local_reference_verification(
            visual_high, visual_edge, audit_config, reference_image_path
        )
        visual_results = await run_llm_inference_on_dropoff_results(
            visual_high, visual_edge, audit_config, reference_image_path=reference_image_path
        )
        results = pd.concat([exact_fast, visual_results], ignore_index=True)
        if not results.empty and "canonical_key" in results:
            results = results.drop_duplicates(subset=["canonical_key"], keep="first")
        results = await _expand_passing_occurrences(results)
        return results.sort_values("relevance_score", ascending=False, na_position="last").reset_index(drop=True)

    ns.update(
        {
            "get_image_mime_type": get_image_mime_type,
            "process_transparency": process_transparency,
            "build_fused_query_text": build_fused_query_text,
            "embed_audit_context": embed_audit_context,
            "generate_audit_config_only": generate_audit_config_only,
            "run_hybrid_search": run_hybrid_search,
            "detect_dropoff_flawless": detect_dropoff_flawless,
            "run_semantic_reranking_and_filter": run_semantic_reranking_and_filter,
            "run_llm_inference_on_dropoff_results": run_llm_inference_on_dropoff_results,
            "run_full_test_bench_pipeline_execution": run_full_test_bench_pipeline_execution,
            # Exposed only for the notebook's offline validation matrix.
            "_local_visual_score_for_validation": _local_visual_score,
            "_audit_search_policy_for_validation": _policy,
            "_visual_candidate_set_for_validation": _visual_candidate_set,
        }
    )
    print("Installed recall/precision overrides. Database remains read-only; no schema changes are used.")


# Activate the readable production implementation for the cells below.
install_recall_precision_overrides(globals())

## Production Integration Contract

The backend integrates **named functions**, not notebook cell numbers:

- `run_visual_audit(goal, reference_image_path, quick_mode)` — stateless one-call production entry point.
- `prepare_notebook_audit(...)` and `execute_notebook_audit(...)` — two-stage interactive Colab workflow.

The notebook state object exists only for interactive calibration. Concurrent App Engine requests should call `run_visual_audit`, which keeps configuration and results local to that request.

### Retrieval, Vision tags, and deduplication

- The system does **not** load or send the database's approximately 70,000 unique Vision tags to Gemini. The audit configuration produces a small relevant vocabulary, and `_bounded_tag_terms(...)` limits the tag query to at most 24 terms. `tag_budget` limits returned asset candidates, not the number of tags inspected in Python.
- Search lanes deduplicate by `content_hash` (falling back to `asset_id` only when a hash is absent) before local verification or Gemini inference. The same bytes are therefore audited once, even if they occur on many pages.
- Accepted canonical matches are expanded in one bounded read-only query to every distinct `(content_hash, page_url)` occurrence. This preserves page-level recall without paying for duplicate image inference. `asset_id` remains the row occurrence identifier; `canonical_match_asset_id` records the audited representative.
- A byte-exact SHA-256 hit is accepted through the exact-hash lane without a Gemini call. Cropped, blurred, resized, or embedded instances cannot use the hash fast path and instead use multimodal retrieval plus bounded local visual rescue before final visual adjudication.
- Quick Mode always selects up to 30 High plus up to 30 Borderline candidates after rescue. Full Mode is intentionally capped and batched for a 3 GB runtime; exhaustive long-running audits should be executed as a background job rather than a single synchronous App Engine request.

> **Recall boundary:** without changing database indexes or storing region-level features, no application-only implementation can mathematically guarantee discovery of a cropped or tiny embedded target that fails to enter every existing retrieval lane. The notebook preserves all retrieved bands, uses a bounded Low-band rescue, and includes labeled quality gates so recall is measured rather than assumed.


In [ ]:
# 7. Production API contract, notebook state, and measurable quality gates
from dataclasses import dataclass
from typing import Any, Dict, Iterable, Optional
import pandas as pd


@dataclass
class NotebookAuditState:
    user_goal: Optional[str] = None
    reference_image_path: Optional[str] = None
    audit_config: Optional[Dict[str, Any]] = None
    results: Optional[pd.DataFrame] = None
    status: str = "idle"
    error: Optional[str] = None


notebook_audit_state = NotebookAuditState()


async def run_visual_audit(user_goal: str, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Stateless production entry point for an async App Engine request handler."""
    if not isinstance(user_goal, str) or not user_goal.strip():
        raise ValueError("user_goal must be a non-empty string")
    config = await generate_audit_config_only(user_goal.strip(), reference_image_path)
    return await run_full_test_bench_pipeline_execution(config, reference_image_path, quick_mode)


async def prepare_notebook_audit(user_goal: str, reference_image_path: Optional[str] = None) -> Dict[str, Any]:
    """Interactive Stage 1; stores exactly what Stage 2 will use."""
    if not isinstance(user_goal, str) or not user_goal.strip():
        raise ValueError("user_goal must be a non-empty string")
    notebook_audit_state.status = "configuring"
    notebook_audit_state.error = None
    try:
        config = await generate_audit_config_only(user_goal.strip(), reference_image_path)
        if not isinstance(config, dict) or not config:
            raise TypeError("Audit configuration generator returned an invalid configuration")
        notebook_audit_state.user_goal = user_goal.strip()
        notebook_audit_state.reference_image_path = reference_image_path
        notebook_audit_state.audit_config = config
        notebook_audit_state.results = None
        notebook_audit_state.status = "configured"
        return config
    except Exception as exc:
        notebook_audit_state.status = "failed"
        notebook_audit_state.error = str(exc)
        raise RuntimeError(f"Stage 1 audit configuration failed: {exc}") from exc


async def execute_notebook_audit(
    audit_config: Optional[Dict[str, Any]] = None,
    reference_image_path: Optional[str] = None,
    quick_mode: bool = False,
) -> pd.DataFrame:
    """Interactive Stage 2 with explicit config and order-safe state fallback."""
    active_config = audit_config or notebook_audit_state.audit_config
    if not isinstance(active_config, dict) or not active_config:
        if notebook_audit_state.user_goal:
            active_config = await prepare_notebook_audit(
                notebook_audit_state.user_goal,
                reference_image_path if reference_image_path is not None else notebook_audit_state.reference_image_path,
            )
        else:
            raise RuntimeError("No audit configuration or cached goal is available. Run Stage 1 or call run_visual_audit(...).")
    active_reference = reference_image_path if reference_image_path is not None else notebook_audit_state.reference_image_path
    notebook_audit_state.status = "running"
    notebook_audit_state.error = None
    try:
        results = await run_full_test_bench_pipeline_execution(active_config, active_reference, quick_mode)
        notebook_audit_state.audit_config = active_config
        notebook_audit_state.reference_image_path = active_reference
        notebook_audit_state.results = results
        notebook_audit_state.status = "completed"
        return results
    except Exception as exc:
        notebook_audit_state.status = "failed"
        notebook_audit_state.error = str(exc)
        raise RuntimeError(f"Stage 2 visual audit failed: {exc}") from exc


def evaluate_audit_quality(
    results_df: pd.DataFrame,
    ground_truth_ids: Iterable[str],
    key_column: str = "asset_id",
) -> Dict[str, float]:
    """Measure final precision/recall/F1 against a labeled set; required for accuracy claims."""
    truth = {str(value) for value in ground_truth_ids}
    if key_column not in results_df.columns:
        raise KeyError(f"Results do not contain key_column={key_column!r}")
    match_mask = results_df.get("matches_criteria", pd.Series(False, index=results_df.index)).fillna(False).astype(bool)
    predicted = {str(value) for value in results_df.loc[match_mask, key_column].dropna()}
    true_positive = len(predicted & truth)
    false_positive = len(predicted - truth)
    false_negative = len(truth - predicted)
    precision = true_positive / (true_positive + false_positive) if predicted else (1.0 if not truth else 0.0)
    recall = true_positive / (true_positive + false_negative) if truth else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "true_positive": float(true_positive),
        "false_positive": float(false_positive),
        "false_negative": float(false_negative),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def assert_quality_gate(report: Dict[str, float], minimum_precision: float, minimum_recall: float) -> None:
    if report["precision"] < minimum_precision or report["recall"] < minimum_recall:
        raise AssertionError(
            f"Quality gate failed: precision={report['precision']:.3f}, recall={report['recall']:.3f}; "
            f"required precision>={minimum_precision:.3f}, recall>={minimum_recall:.3f}"
        )


async def close_visual_audit_resources() -> None:
    """Call from the backend's shutdown hook."""
    await close_alloydb_connection()
    client.close()

## Offline Validation Matrix — No AlloyDB or Gemini Calls

This matrix validates routing, memory caps, Quick Mode quotas, batching, and deterministic reranking. OpenCV visual fixtures run when `opencv-python-headless` is installed.


In [ ]:
# 8. Offline validation matrix — runs locally only (no AlloyDB, GCS, or Gemini calls)
# Run the production runtime cell first.
import asyncio
import io
import importlib.util
import time

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter

required_helpers = (
    "evaluate_audit_quality",
    "assert_quality_gate",
    "_local_visual_score_for_validation",
    "_audit_search_policy_for_validation",
    "_visual_candidate_set_for_validation",
    "process_transparency",
    "build_fused_query_text",
    "run_semantic_reranking_and_filter",
    "run_llm_inference_on_dropoff_results",
)
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        "Run the production runtime cell first. Missing helpers: " + ", ".join(missing_helpers)
    )

validation_rows = []

def add_result(use_case, status, observation):
    validation_rows.append(
        {"Use case": use_case, "Status": status, "Observation": observation}
    )

def png_bytes(image):
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()

# A deliberately feature-rich synthetic reference mark. It supports both template and
# keypoint verification, without using any real user image or database object.
reference = Image.new("RGB", (360, 240), "white")
draw = ImageDraw.Draw(reference)
draw.rounded_rectangle((14, 14, 346, 226), radius=22, outline="#1A73E8", width=10)
draw.ellipse((52, 58, 172, 178), fill="#EA4335", outline="#202124", width=5)
draw.polygon([(220, 58), (312, 118), (220, 178)], fill="#34A853", outline="#202124")
draw.line((192, 54, 192, 184), fill="#FBBC04", width=9)
draw.text((104, 191), "AUDIT", fill="#202124")
reference_bytes = png_bytes(reference)

# Visual transformations that represent the retrieval/audit cases we care about.
resized = reference.resize((540, 360))
embedded = Image.new("RGB", (960, 640), "#F1F3F4")
embedded.paste(reference.resize((151, 101)), (745, 482))
cropped = reference.crop((24, 18, 334, 222))
blurred = reference.filter(ImageFilter.GaussianBlur(radius=2.8))
unrelated = Image.new("RGB", (720, 480), "#202124")
unrelated_draw = ImageDraw.Draw(unrelated)
unrelated_draw.rectangle((70, 70, 650, 410), fill="#8AB4F8")
unrelated_draw.ellipse((280, 150, 470, 340), fill="#FBBC04")

if importlib.util.find_spec("cv2") is None:
    for label in ("Exact image-to-image", "Resized near-duplicate", "Small embedded target", "Cropped target", "Blurred target", "Unrelated negative"):
        add_result(label, "SKIPPED", "OpenCV is unavailable. Run the installation cell, then rerun this matrix.")
else:
    reference_rgb = np.asarray(reference.convert("RGB"))
    visual_cases = [
        ("Exact image-to-image", reference_bytes, 0.90),
        ("Resized near-duplicate", png_bytes(resized), 0.70),
        ("Small embedded target", png_bytes(embedded), 0.45),
        ("Cropped target", png_bytes(cropped), 0.45),
        ("Blurred target", png_bytes(blurred), 0.45),
        ("Unrelated negative", png_bytes(unrelated), None),
    ]
    visual_evidence = {}
    for label, candidate_bytes, threshold in visual_cases:
        evidence = _local_visual_score_for_validation(reference_rgb, candidate_bytes)
        visual_evidence[label] = evidence
        score = float(evidence["local_visual_score"])
        method = evidence["local_visual_method"]
        if threshold is None:
            exact_score = float(visual_evidence["Exact image-to-image"]["local_visual_score"])
            status = "PASS" if score < exact_score else "CHECK"
            add_result(label, status, f"score={score:.3f}; method={method}; exact baseline={exact_score:.3f}")
        else:
            status = "PASS" if score >= threshold else "CHECK"
            add_result(label, status, f"score={score:.3f}; method={method}; expected local signal ≥ {threshold:.2f}")

# Verify lossless PNG output, alpha compositing, and active-payload memory capping.
transparent_large = Image.new("RGBA", (2300, 1800), (0, 0, 0, 0))
transparent_draw = ImageDraw.Draw(transparent_large)
transparent_draw.ellipse((300, 250, 2000, 1550), fill=(255, 255, 255, 190))
processed = process_transparency(png_bytes(transparent_large))
with Image.open(io.BytesIO(processed)) as processed_image:
    output_is_png = processed_image.format == "PNG"
    output_is_bounded = max(processed_image.size) <= 1600
add_result(
    "Transparent / oversized asset",
    "PASS" if output_is_png and output_is_bounded else "FAIL",
    f"normalized to PNG; max side ≤ 1600 px = {output_is_bounded}",
)

# Validate goal routing without asking Gemini to create the configuration.
policy_for_validation = _audit_search_policy_for_validation
exact_policy = policy_for_validation(
    {"audit_goal": "Find the exact uploaded profile picture everywhere it occurs."}, True
)
legacy_policy = policy_for_validation(
    {"audit_goal": "Find the old solid brand mark and reject the newer gradient version."}, True
)
text_only_policy = policy_for_validation(
    {"audit_goal": "Find product screenshots containing checkout buttons."}, False
)
policy_ok = (
    exact_policy["mode"] == "exact_or_near_duplicate"
    and exact_policy["local_verification_budget"] > 0
    and legacy_policy["mode"] == "version_sensitive"
    and legacy_policy["color_is_discriminating"]
    and text_only_policy["mode"] == "semantic_visual"
    and text_only_policy["local_verification_budget"] == 0
)
add_result(
    "Exact, legacy/color, and text-only routing",
    "PASS" if policy_ok else "FAIL",
    f"exact={exact_policy['mode']}; legacy={legacy_policy['mode']}; text-only={text_only_policy['mode']}",
)

# The App Engine profile must clamp caller-supplied concurrency rather than trusting it.
overloaded_policy = policy_for_validation(
    {
        "audit_goal": "Find the exact uploaded profile picture everywhere it occurs.",
        "llm_concurrency": 32,
        "llm_batch_size": 32,
        "local_verification_concurrency": 32,
    },
    True,
)
runtime_guardrail_ok = (
    exact_policy.get("runtime_profile") == "app_engine_3gb"
    and overloaded_policy["llm_concurrency"] == 2
    and overloaded_policy["llm_batch_size"] == 4
    and overloaded_policy["local_verification_concurrency"] == 2
)
add_result(
    "3 GB App Engine live-work cap",
    "PASS" if runtime_guardrail_ok else "FAIL",
    f"LLM workers={overloaded_policy['llm_concurrency']}; batch={overloaded_policy['llm_batch_size']}; local workers={overloaded_policy['local_verification_concurrency']}",
)

legacy_query = build_fused_query_text(
    {
        "audit_goal": "Find the old solid brand mark and reject the newer gradient version.",
        "inclusion_criteria": ["Use the old solid color treatment."],
        "_search_policy": legacy_policy,
    }
)
add_result(
    "Solid-versus-gradient preservation",
    "PASS" if "solid-versus-gradient" in legacy_query else "FAIL",
    "The fused retrieval query preserves color/style as match evidence rather than globally ignoring it.",
)

# Quick Mode must retain its shape: 30 High + 30 Borderline, never a pooled top 60.
quick_high_input = pd.DataFrame(
    [{"asset_id": f"high-{index}", "canonical_key": f"high-{index}", "relevance_score": 100 - index, "segmentation_band": "high"} for index in range(35)]
)
quick_edge_input = pd.DataFrame(
    [{"asset_id": f"edge-{index}", "canonical_key": f"edge-{index}", "relevance_score": 50 - index, "segmentation_band": "borderline"} for index in range(35)]
)
quick_high, quick_edge = _visual_candidate_set_for_validation(
    quick_high_input, quick_edge_input, pd.DataFrame(), {"_search_policy": exact_policy}, quick_mode=True
)
quick_quota_ok = (
    len(quick_high) == 30
    and len(quick_edge) == 30
    and quick_high.iloc[0]["asset_id"] == "high-0"
    and quick_edge.iloc[0]["asset_id"] == "edge-0"
)
add_result(
    "Quick Mode 30 High + 30 Borderline quota",
    "PASS" if quick_quota_ok else "FAIL",
    f"selected High={len(quick_high)} and Borderline={len(quick_edge)}; bands are not pooled.",
)

# This confirms the legacy description-only LLM reranker is bypassed before final vision audit.
rerank_high = pd.DataFrame([{"asset_id": "a", "relevance_score": 0.5}, {"asset_id": "b", "relevance_score": 0.9}])
rerank_edge = pd.DataFrame([{"asset_id": "c", "relevance_score": 0.4}])
reranked_high, reranked_edge, reranked_low = await run_semantic_reranking_and_filter(
    rerank_high, rerank_edge, {"audit_goal": "offline validation"}
)
rerank_ok = (
    len(reranked_high) == 2
    and len(reranked_edge) == 1
    and reranked_low.empty
    and reranked_high.iloc[0]["asset_id"] == "b"
)
add_result(
    "Description-only LLM reranker bypass",
    "PASS" if rerank_ok else "FAIL",
    "Candidate ranks are retained deterministically; no remote cross-encoder is invoked.",
)

# Exercise final-audit batching using a local coroutine. It deliberately replaces the
# final vision call only for this test and restores the original function immediately.
concurrency = {"active": 0, "peak": 0}
original_single_audit = globals().get("run_llm_audit_single")

async def _validation_single_audit(asset, audit_config, _executor=None, reference_image_part=None):
    concurrency["active"] += 1
    concurrency["peak"] = max(concurrency["peak"], concurrency["active"])
    await asyncio.sleep(0.01)
    concurrency["active"] -= 1
    return {
        **asset,
        "matches_criteria": False,
        "match_confidence": 0,
        "match_rationale": "Local bounded-concurrency validation only.",
    }

globals()["run_llm_audit_single"] = _validation_single_audit
try:
    bounded_candidates = pd.DataFrame(
        [{"asset_id": f"fixture-{index}", "relevance_score": 1.0 - index / 20} for index in range(11)]
    )
    bounded_results = await run_llm_inference_on_dropoff_results(
        bounded_candidates,
        pd.DataFrame(),
        {"llm_concurrency": 3, "llm_batch_size": 4},
        reference_image_path=None,
    )
    bounded_ok = len(bounded_results) == 11 and concurrency["peak"] <= 3
    add_result(
        "Bounded final visual-audit work",
        "PASS" if bounded_ok else "FAIL",
        f"audited={len(bounded_results)} fixtures; observed concurrent tasks={concurrency['peak']} (cap=3)",
    )
finally:
    if original_single_audit is None:
        globals().pop("run_llm_audit_single", None)
    else:
        globals()["run_llm_audit_single"] = original_single_audit


# Quality metrics must be computed from labels, not inferred from confidence.
quality_fixture = pd.DataFrame([
    {"asset_id": "a", "matches_criteria": True},
    {"asset_id": "b", "matches_criteria": True},
    {"asset_id": "x", "matches_criteria": False},
])
quality_report = evaluate_audit_quality(quality_fixture, {"a", "b", "c"})
quality_ok = abs(quality_report["precision"] - 1.0) < 1e-9 and abs(quality_report["recall"] - (2 / 3)) < 1e-9
add_result(
    "Precision/recall metric calibration (intentional missed label)",
    "PASS" if quality_ok else "FAIL",
    f"precision={quality_report['precision']:.3f}; recall={quality_report['recall']:.3f}; F1={quality_report['f1']:.3f}",
)

# Regression for the corrected chord projection used by drop-off segmentation.
dropoff_fixture = pd.DataFrame({
    "asset_id": [f"d-{i}" for i in range(80)],
    "relevance_score": np.r_[np.linspace(1.0, 0.82, 20), np.linspace(0.55, 0.30, 30), np.linspace(0.08, 0.01, 30)],
    "vector_distance": np.linspace(0.4, 1.0, 80),
    "promoted_by_keyword_or_tag": False,
})
drop_high, drop_edge, drop_low = detect_dropoff_flawless(dropoff_fixture)
dropoff_ok = len(drop_high) > 0 and len(drop_edge) > 0 and len(drop_low) > 0 and len(drop_high) + len(drop_edge) + len(drop_low) == 80
add_result(
    "Drop-off partition completeness",
    "PASS" if dropoff_ok else "FAIL",
    f"High={len(drop_high)}; Borderline={len(drop_edge)}; Low reserve={len(drop_low)}; total preserved={len(drop_high)+len(drop_edge)+len(drop_low)}",
)

validation_df = pd.DataFrame(validation_rows)
print("Offline validation completed. No AlloyDB, GCS, Gemini, or database-write calls were made.")
try:
    from IPython.display import display
    display(validation_df)
except Exception:
    print(validation_df.to_string(index=False))

failed_validations = validation_df[validation_df["Status"] != "PASS"]
if not failed_validations.empty:
    raise AssertionError("Offline validation failures:\n" + failed_validations.to_string(index=False))
print(f"All {len(validation_df)} offline validation checks passed.")

In [ ]:
# TEST RUN — Stage 1: generate and inspect the audit configuration
import json

TEST_AUDIT_GOAL = "Find all pages with this exact image"
UPLOAD_REFERENCE_IMAGE = True
QUICK_MODE = True

reference_image_path = globals().get("reference_image_path")
if UPLOAD_REFERENCE_IMAGE:
    try:
        from google.colab import files
        print("[OPTIONAL] Upload a reference image. Cancel for a text-only audit:")
        uploaded = files.upload()
        if uploaded:
            reference_image_path = next(iter(uploaded))
            print(f"Reference image selected: {reference_image_path}")
        elif reference_image_path:
            print(f"No new upload; retaining: {reference_image_path}")
        else:
            print("No reference image selected; running a text-only Stage 1.")
    except ImportError:
        print("Colab upload is unavailable; using reference_image_path if already configured.")

notebook_audit_state.user_goal = TEST_AUDIT_GOAL
notebook_audit_state.reference_image_path = reference_image_path
audit_config_global = await prepare_notebook_audit(TEST_AUDIT_GOAL, reference_image_path)
print("Stage 1 completed. This exact configuration will be used by Stage 2:")
print(json.dumps(audit_config_global, indent=2, default=str))

In [ ]:
# TEST RUN — Stage 2: execute retrieval, recall rescue, and bounded visual audit
# You may edit audit_config_global before this call to calibrate the generated criteria.

active_config = globals().get("audit_config_global")
active_reference = globals().get("reference_image_path")
active_goal = str(globals().get("TEST_AUDIT_GOAL", "Find all pages with this exact image")).strip()
quick_mode = bool(globals().get("QUICK_MODE", True))

if not isinstance(active_config, dict) or not active_config:
    if not active_goal:
        raise RuntimeError("Set TEST_AUDIT_GOAL or call run_visual_audit(user_goal, reference_image_path).")
    print("Stage 1 state is absent (for example after a runtime restart); regenerating it now.")
    notebook_audit_state.user_goal = active_goal
    notebook_audit_state.reference_image_path = active_reference
    active_config = await prepare_notebook_audit(active_goal, active_reference)
    audit_config_global = active_config

df_results_global = await execute_notebook_audit(active_config, active_reference, quick_mode)
print(
    f"Stage 2 completed: {len(df_results_global):,} result rows; "
    f"session_status={notebook_audit_state.status}; quick_mode={quick_mode}"
)
df_results_global